In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: HSV ROADWAY MASK GENERATION
    # ==============================================================================
    # Discard foliage/off-road noise targeting asphalt HSV bounds
    lower_road_hsv = np.array([0, 0, 40])
    upper_road_hsv = np.array([180, 50, 200])

    # Binary mask: 255 for asphalt surface, 0 for off-road/trees
    road_mask = cv2.inRange(hsv, lower_road_hsv, upper_road_hsv)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STEP 1: LAPLACIAN EDGE DETECTION APPLIED STRICTLY TO ROADWAY
    # ==============================================================================
    blurred = cv2.GaussianBlur(gray_img, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Restrict edge map strictly within HSV road mask
    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)
    laplacian_binary = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"HSV ROADWAY ISOLATION & EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: HSV Color Space Image
    axes_prep[1].imshow(hsv_display_rgb)
    axes_prep[1].set_title("2. HSV Color Space Image")
    axes_prep[1].axis('off')

    # Column 3: Edge Detection (Roadway Only)
    axes_prep[2].imshow(laplacian_binary, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Roadway Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & SINUSOIDAL INTERCEPTION RANKING
    # ==============================================================================
    # Compute standard Hough Transform (rho = x*cos(theta) + y*sin(theta))
    # Returns polar lines sorted by accumulator strength (number of coincident curve intersections)
    rho_res = 1              # Distance resolution in pixels
    theta_res = np.pi / 180  # Angular resolution in radians
    hough_threshold = 30     # Accumulator threshold

    lines_polar = cv2.HoughLines(laplacian_binary, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    # Function to overlay polar Hough lines onto an image
    def draw_polar_lines(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        # Clamp range indices based on total lines found
        actual_start = min(start_rank, len(lines))
        actual_end = min(end_rank, len(lines))
        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            # Calculate line endpoints across image boundary
            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw infinite Hough polar candidate line in Red
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    # Stage 1: Ranks 10 to 20 Coincidence Points
    overlay_10_20, count_10_20 = draw_polar_lines(img_rgb, lines_polar, start_rank=10, end_rank=20)

    # Stage 2: Ranks 30 to 50 Coincidence Points
    overlay_30_50, count_30_50 = draw_polar_lines(img_rgb, lines_polar, start_rank=30, end_rank=50)

    # Stage 3: Ranks 60 to 70 Coincidence Points
    overlay_60_70, count_60_70 = draw_polar_lines(img_rgb, lines_polar, start_rank=60, end_rank=70)

    # ==============================================================================
    # DISPLAY POLAR HOUGH TRANSFORM RANKED INTERSECTION STAGES
    # ==============================================================================
    fig_hough, axes_hough = plt.subplots(1, 3, figsize=(21, 6))
    fig_hough.suptitle(f"POLAR HOUGH TRANSFORM - INTERSECTION RANKINGS ({filename})", fontsize=16, weight='bold')

    # Output 1: Ranks 10 to 20
    axes_hough[0].imshow(overlay_10_20)
    axes_hough[0].set_title(f"Rank 10 to 20 Intersections\n(Lines Drawn: {count_10_20})", fontsize=12, weight='bold')
    axes_hough[0].axis('off')

    # Output 2: Ranks 30 to 50
    axes_hough[1].imshow(overlay_30_50)
    axes_hough[1].set_title(f"Rank 30 to 50 Intersections\n(Lines Drawn: {count_30_50})", fontsize=12, weight='bold')
    axes_hough[1].axis('off')

    # Output 3: Ranks 60 to 70
    axes_hough[2].imshow(overlay_60_70)
    axes_hough[2].set_title(f"Rank 60 to 70 Intersections\n(Lines Drawn: {count_60_70})", fontsize=12, weight='bold')
    axes_hough[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: HSV ROADWAY MASK GENERATION
    # ==============================================================================
    lower_road_hsv = np.array([0, 0, 40])
    upper_road_hsv = np.array([180, 50, 200])

    # Binary mask: 255 for asphalt surface, 0 for off-road/trees
    road_mask = cv2.inRange(hsv, lower_road_hsv, upper_road_hsv)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STEP 1: LAPLACIAN EDGE DETECTION APPLIED STRICTLY TO ROADWAY
    # ==============================================================================
    blurred = cv2.GaussianBlur(gray_img, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Restrict edge map strictly within HSV road mask
    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)
    laplacian_binary = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"HSV ROADWAY ISOLATION & EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(hsv_display_rgb)
    axes_prep[1].set_title("2. HSV Color Space Image")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_binary, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Roadway Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & RANKED INTERSECTION SLICES
    # ==============================================================================
    rho_res = 1              # Distance resolution in pixels
    theta_res = np.pi / 180  # Angular resolution in radians
    hough_threshold = 20     # Lower threshold to collect sufficient candidate lines

    # Compute Standard Polar Hough Lines sorted by accumulator interception strength
    lines_polar = cv2.HoughLines(laplacian_binary, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    # Function to overlay a specific rank range of polar Hough lines
    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        # Adjust start and end indices based on available total detected lines
        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            # Project line endpoints across image boundaries
            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw polar line candidate in red
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    # Define the 5 rank slice ranges requested
    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    # Generate overlay images for each rank bracket
    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"POLAR HOUGH TRANSFORM INTERSECTION RANKINGS 1 TO 100 ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STAGE 0: HSV DARK/GREY ASPHALT MASK (EXCLUDING BRIGHT WHITE LANE LINES)
    # ==============================================================================
    # Target dark/grey asphalt pixels specifically:
    # S: 0-40 (low saturation removes yellow foliage/colored marks)
    # V: 40-140 (restricts brightness to dark/grey asphalt, filtering out bright white lines >180)
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])

    # Binary mask: 255 for dark/grey asphalt surface, 0 for white lines & foliage
    asphalt_crack_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # ==============================================================================
    # STEP 1: LAPLACIAN EDGE DETECTION APPLIED STRICTLY TO DARK GREY ASPHALT
    # ==============================================================================
    # Enhance local subtle crack contrasts using CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Apply thresholding to catch fine crack boundaries
    _, laplacian_binary = cv2.threshold(laplacian_norm, 25, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly within the dark/grey asphalt mask (ignoring white lines)
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=asphalt_crack_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"DARK/GREY ROADWAY ISOLATION & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: HSV Color Space Image
    axes_prep[1].imshow(hsv_display_rgb)
    axes_prep[1].set_title("2. HSV Color Space Image")
    axes_prep[1].axis('off')

    # Column 3: Edge Map Restricted to Dark/Grey Asphalt
    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Dark Grey Road Cracks Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1              # Distance resolution in pixels
    theta_res = np.pi / 180  # Angular resolution in radians
    hough_threshold = 15     # Accumulator threshold tuned for fine crack lines

    # Compute Standard Polar Hough Lines sorted by accumulator intersection strength
    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    # Function to overlay a specific rank slice range of polar Hough lines
    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw crack candidate lines in Red
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    # Define the 5 rank slice ranges
    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"CRACK LINE DETECTIONS BY HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STAGE 0: HSV DARK/GREY ASPHALT MASK (EXCLUDING WHITE LANES & SHOULDER EDGES)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # Crop outer left/right guardrail boundaries (10% side margins)
    margin_mask = np.zeros((h, w), dtype=np.uint8)
    margin_mask[:, int(w * 0.10):int(w * 0.90)] = 255
    asphalt_mask = cv2.bitwise_and(asphalt_mask, margin_mask)

    # ==============================================================================
    # STEP 1: LAPLACIAN + GRADIENT ORIENTATION FILTERING (SUPPRESS VERTICAL DIVIDERS)
    # ==============================================================================
    # Enhance local contrast for fine pavement cracks
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 25, 255, cv2.THRESH_BINARY)
    laplacian_binary = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=asphalt_mask)

    # Calculate Sobel gradients to filter out long vertical road boundaries
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Calculate edge orientation angles in degrees (0 to 180)
    angle = np.arctan2(np.abs(sobel_y), np.abs(sobel_x)) * (180.0 / np.pi)

    # Keep edges that have horizontal/transverse components (suppress pure vertical road lines)
    non_vertical_mask = (angle > 15) & (angle < 165)

    # Final cleaned crack edge map
    laplacian_cracks_only = np.zeros_like(laplacian_binary)
    laplacian_cracks_only[(laplacian_binary == 255) & non_vertical_mask] = 255

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ASPHALT CRACKS EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(hsv_display_rgb)
    axes_prep[1].set_title("2. HSV Color Space Image")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_cracks_only, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Transverse/Pavement Cracks Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPT RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(laplacian_cracks_only, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"PAVEMENT CRACK HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: HSV ASPHALT MASK & VEGETATION SUPPRESSION MASK
    # ==============================================================================
    # 1. Target dark/grey asphalt pixels specifically
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([40, 40, 180])
    asphalt_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # 2. Vegetation Suppression Mask: Target green/yellow foliage (Hue ~ 15 to 85)
    # and remove them from the processing pipeline
    lower_veg = np.array([15, 40, 40])
    upper_veg = np.array([85, 255, 255])
    veg_mask = cv2.inRange(hsv, lower_veg, upper_veg)
    non_veg_mask = cv2.bitwise_not(veg_mask)

    # Combine asphalt mask with vegetation suppression
    road_and_veg_suppressed_mask = cv2.bitwise_and(asphalt_mask, non_veg_mask)

    # Create visual representation for Column 2 showing vegetation suppressed (masked out to black)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    hsv_display_suppressed = cv2.bitwise_and(hsv_display_rgb, hsv_display_rgb, mask=non_veg_mask)

    # ==============================================================================
    # STEP 1: LAPLACIAN EDGE DETECTION APPLIED STRICTLY TO SUPPRESSED ASPHALT
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 25, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly within the vegetation-suppressed asphalt mask
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=road_and_veg_suppressed_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"VEGETATION SUPPRESSION & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: HSV Color Space with Vegetation Suppressed
    axes_prep[1].imshow(hsv_display_suppressed)
    axes_prep[1].set_title("2. HSV Color (Vegetation Suppressed)")
    axes_prep[1].axis('off')

    # Column 3: Edge Map Restricted to Clean Roadway Cracks
    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cleaned Cracks Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    # Compute Standard Polar Hough Lines on the vegetation-suppressed edge map
    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"VEGETATION-SUPPRESSED HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STAGE 0: STRICT ROADWAY ROI & VEGETATION SUPPRESSION MASK
    # ==============================================================================
    # 1. Target dark/grey asphalt pixels specifically
    lower_dark_grey_asphalt = np.array([0, 0, 0])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # 2. Vegetation Suppression Mask (Green/Yellow + Dark Shadowy Bushes on Sides)
    lower_veg = np.array([10, 20, 20])
    upper_veg = np.array([90, 255, 255])
    veg_mask = cv2.inRange(hsv, lower_veg, upper_veg)
    non_veg_mask = cv2.bitwise_not(veg_mask)

    # 3. Strict Geometric Roadway Width Mask (Zeroes out side bushes completely)
    # Adjust column boundaries (e.g., keeping from 15% to 85% of image width)
    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.15), 0), (int(w * 0.85), h), 255, -1)

    # Combine all filters to isolate ONLY the clean road surface
    final_road_mask = cv2.bitwise_and(asphalt_mask, non_veg_mask)
    final_road_mask = cv2.bitwise_and(final_road_mask, road_roi_mask)

    # Create visual representation for Column 2 with side bushes completely zeroed out (black)
    hsv_display_suppressed = cv2.bitwise_and(hsv_display_rgb, hsv_display_rgb, mask=road_roi_mask)

    # ==============================================================================
    # STEP 1: LAPLACIAN EDGE DETECTION APPLIED STRICTLY TO THE CLEANED ROAD ROI
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 25, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly to the fully cleaned road mask
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=final_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"STRICT SIDE BUSH ZEROING & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: HSV Color with Side Bushes Completely Zeroed Out
    axes_prep[1].imshow(hsv_display_suppressed)
    axes_prep[1].set_title("2. HSV Color (Side Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Edge Map Restricted to Clean Roadway Cracks
    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Roadway Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"ROAD-ISOLATED HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    # Target dark/grey asphalt pixels specifically (ignoring bright lane lines >140 and bushes)
    lower_dark_grey_asphalt = np.array([0, 0, 10])
    upper_dark_grey_asphalt = np.array([10, 0, 100])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # Strict geometric roadway window (keeps middle highway region, zeroes out side bushes completely)
    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    # Final binary road mask: Road pixels are 255 (1), everything else is strictly 0
    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)

    # Column 2 display: Show original colors ONLY inside the road mask (background set to absolute zero/black)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=asphalt_color_mask)
    # ==============================================================================
    # STEP 1: HIGH-CONTRAST LAPLACIAN EDGE DETECTION FOR CRACKS ONLY
    # ==============================================================================
    # Enhance subtle crack contrast using CLAHE
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    # Gentle blur to reduce rough asphalt grain while preserving crack edges
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian second-derivative filter
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold to isolate sharp crack edges
    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly inside the binary road mask (eliminating lane line edges and side bushes)
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: Road Region Only (Background & Bushes strictly 0 / Black)
    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Clear Pavement Cracks Edge Map
    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 100

    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"ROAD-ISOLATED HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    # Target dark/grey asphalt pixels specifically (ignoring bright lane lines >140 and bushes)
    lower_dark_grey_asphalt = np.array([0, 0, 0])
    upper_dark_grey_asphalt = np.array([40, 0 , 180])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # Strict geometric roadway window (keeps middle highway region, zeroes out side bushes completely)
    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    # Final binary road mask: Road pixels are 255 (1), everything else is strictly 0
    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)

    # Column 2 display: Show original colors ONLY inside the road mask (background set to absolute zero/black)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=asphalt_color_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: Road Region Only (Background & Bushes strictly 0 / Black)
    axes_prep[1].imshow(asphalt_color_mask)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Clear Pavement Cracks Edge Map
    axes_prep[2].imshow(binary_road_mask, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()


HSV CHANNEL BREAKDOWN

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# STANDALONE SCRIPT: DISPLAY INDIVIDUAL HSV CHANNELS SEPARATELY
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Convert BGR to HSV color space
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Split into individual Hue, Saturation, and Value channels
    h_channel, s_channel, v_channel = cv2.split(hsv_img)

    # Plot and display the channels side-by-side
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"HSV Channel Breakdown ({filename})", fontsize=16, weight='bold')

    # 1. Original Image
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB Image")
    axes[0].axis('off')

    # 2. Hue Channel
    axes[1].imshow(h_channel, cmap='gray')
    axes[1].set_title("Hue (H) Channel")
    axes[1].axis('off')

    # 3. Saturation Channel
    axes[2].imshow(s_channel, cmap='gray')
    axes[2].set_title("Saturation (S) Channel")
    axes[2].axis('off')

    # 4. Value Channel
    axes[3].imshow(v_channel, cmap='gray')
    axes[3].set_title("Value (V) Channel")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# STANDALONE SCRIPT: SATURATION BINARY INVERSION & BITWISE AND WITH RGB
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Convert BGR to HSV color space
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # 3. Extract Saturation (S) component
    _, s_channel, _ = cv2.split(hsv_img)

    # 4. Convert Saturation component into a binary image via thresholding
    # Adjust threshold value (e.g., 50) depending on your specific image contrast
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # 5. Inverse the binary mask
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 6. Apply bitwise_and between the inverted binary mask and the original RGB image
    # Note: cv2.bitwise_and works with 3-channel images if the mask is single-channel
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 7. Plot and display the steps side-by-side
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"Saturation Inversion & Bitwise Masking ({filename})", fontsize=16, weight='bold')

    # Original RGB Image
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB")
    axes[0].axis('off')

    # Saturation Binary (Thresholded)
    axes[1].imshow(s_binary, cmap='gray')
    axes[1].set_title("Saturation Binary")
    axes[1].axis('off')

    # Inverted Saturation Binary
    axes[2].imshow(s_binary_inv, cmap='gray')
    axes[2].set_title("Inverted Saturation Binary")
    axes[2].axis('off')

    # Bitwise AND Result with RGB Image
    axes[3].imshow(rgb_masked_output)
    axes[3].set_title("Bitwise AND Output")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 255/1, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: SOBEL EDGE DETECTION APPLIED STRICTLY TO THE ROADWAY
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Sobel gradients in X and Y directions
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Combine X and Y gradients to get total magnitude
    sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_magnitude)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold to isolate crack edges
    _, sobel_binary = cv2.threshold(sobel_norm, 25, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly inside the binary road mask
    sobel_binary_cracks = cv2.bitwise_and(sobel_binary, sobel_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ROADWAY ISOLATION & SOBEL EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: Road Region Only (Background & Bushes zeroed to black)
    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Sobel Edge Map
    axes_prep[2].imshow(sobel_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Sobel Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(sobel_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"SOBEL-BASED HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv_display_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    # ==============================================================================
    # STAGE 0: HSV DARK/GREY ASPHALT MASK (EXCLUDING WHITE LANES & SIDE MARGINS)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # Crop outer left/right guardrail boundaries (10% side margins)
    margin_mask = np.zeros((h, w), dtype=np.uint8)
    margin_mask[:, int(w * 0.10):int(w * 0.90)] = 255
    asphalt_mask = cv2.bitwise_and(asphalt_mask, margin_mask)

    # ==============================================================================
    # STEP 1: SOBEL X & Y GRADIENT MAGNITUDE & ORIENTATION FILTERING
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute individual Sobel X and Y gradients
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Calculate overall gradient magnitude
    sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_magnitude)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold gradient magnitude to catch crack edges
    _, sobel_binary = cv2.threshold(sobel_norm, 25, 255, cv2.THRESH_BINARY)
    sobel_binary = cv2.bitwise_and(sobel_binary, sobel_binary, mask=asphalt_mask)

    # Calculate edge orientation angles in degrees (0 to 180)
    angle = np.arctan2(np.abs(sobel_y), np.abs(sobel_x)) * (180.0 / np.pi)

    # Keep edges with transverse components (suppressing pure vertical road lines)
    non_vertical_mask = (angle > 15) & (angle < 165)

    # Final cleaned crack edge map using Sobel
    sobel_cracks_only = np.zeros_like(sobel_binary)
    sobel_cracks_only[(sobel_binary == 255) & non_vertical_mask] = 255

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ASPHALT CRACKS - SOBEL EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(hsv_display_rgb)
    axes_prep[1].set_title("2. HSV Color Space Image")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(sobel_cracks_only, cmap='gray')
    axes_prep[2].set_title("3. Sobel Edge Map (Exact Cracks Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPT RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(sobel_cracks_only, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"SOBEL CRACK HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: SOBEL X & Y GRADIENT EDGE DETECTION FOR CRACKS ONLY
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Sobel gradients in X and Y directions
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Combine X and Y gradients to calculate overall magnitude
    sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_magnitude)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold to isolate sharp crack edges
    _, sobel_binary = cv2.threshold(sobel_norm, 25, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly inside the binary road mask
    sobel_binary_cracks = cv2.bitwise_and(sobel_binary, sobel_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & SOBEL CRACK DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: Road Region Only (Background & Bushes strictly 0 / Black)
    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Clear Pavement Cracks Edge Map via Sobel X & Y
    axes_prep[2].imshow(sobel_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Sobel Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(sobel_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"SOBEL-BASED ROAD-ISOLATED HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: INDIVIDUAL SOBEL X, SOBEL Y, AND COMBINED EDGE DETECTION
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # 1. Sobel X (Horizontal Gradient)
    sobel_x_64f = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_x_abs = cv2.convertScaleAbs(sobel_x_64f)
    sobel_x_norm = cv2.normalize(sobel_x_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_x_binary = cv2.threshold(sobel_x_norm, 25, 255, cv2.THRESH_BINARY)
    sobel_x_display = cv2.bitwise_and(sobel_x_binary, sobel_x_binary, mask=binary_road_mask)

    # 2. Sobel Y (Vertical Gradient)
    sobel_y_64f = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_y_abs = cv2.convertScaleAbs(sobel_y_64f)
    sobel_y_norm = cv2.normalize(sobel_y_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_y_binary = cv2.threshold(sobel_y_norm, 25, 255, cv2.THRESH_BINARY)
    sobel_y_display = cv2.bitwise_and(sobel_y_binary, sobel_y_binary, mask=binary_road_mask)

    # 3. Combined Magnitude (Sobel X + Sobel Y Combined)
    sobel_magnitude = cv2.magnitude(sobel_x_64f, sobel_y_64f)
    sobel_mag_abs = cv2.convertScaleAbs(sobel_magnitude)
    sobel_mag_norm = cv2.normalize(sobel_mag_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_mag_binary = cv2.threshold(sobel_mag_norm, 25, 255, cv2.THRESH_BINARY)
    sobel_binary_cracks = cv2.bitwise_and(sobel_mag_binary, sobel_mag_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES (SHOWING X, Y, AND COMBINED)
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 5, figsize=(25, 5))
    fig_prep.suptitle(f"SOBEL X, SOBEL Y & COMBINED EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road ROI Masked")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(sobel_x_display, cmap='gray')
    axes_prep[2].set_title("3. Sobel X (Horizontal)")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(sobel_y_display, cmap='gray')
    axes_prep[3].set_title("4. Sobel Y (Vertical)")
    axes_prep[3].axis('off')

    axes_prep[4].imshow(sobel_binary_cracks, cmap='gray')
    axes_prep[4].set_title("5. Combined Magnitude")
    axes_prep[4].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & ACCUMULATOR INTERCEPTION RANKINGS
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(sobel_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range(base_img, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)
            count += 1

        return overlay, count

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn = draw_polar_line_range(img_rgb, lines_polar, start_rank=start, end_rank=end)
        slice_results.append((start, end, img_out, lines_drawn))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"COMBINED SOBEL HOUGH RANKINGS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        axes_ranks[idx].set_title(f"Rank {start} to {end}\n(Lines: {lines_drawn})", fontsize=12, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: HIGH-CONTRAST LAPLACIAN EDGE DETECTION FOR CRACKS ONLY
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & PIXEL MEASUREMENT EXTRACTION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw line on display image
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            # Calculate actual line segment length within image bounds using intersection
            # Simplified pixel length approximation of the chord inside image bounds
            # Or compute Euclidean length of segment bounded by image limits:
            img_h, img_w = base_img.shape[:2]

            # Find clipping points or use line length metric based on active edge pixels near the line
            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            # Count active crack pixels intersecting/associated with this line
            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                # Length as max distance between active pixels on this line
                pts = active_pixels[:, [1, 0]] # x, y
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else np.linalg.norm([x2-x1, y2-y1])
            else:
                line_len = 0.0

            # Estimate local crack thickness (local contour/distance transform or area/length ratio)
            # Thickness can be approximated as area of local component divided by length
            if line_len > 0:
                line_area = np.sum(intersection > 0)
                line_thick = line_area / line_len
            else:
                line_thick = 0.0

            total_length += line_len
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_binary_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_thick))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES WITH PIXEL METRICS
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"ROAD-ISOLATED HOUGH RANKINGS & CRACK METRICS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_str = (f"Rank {start}-{end}\n"
                     f"Lines: {lines_drawn}\n"
                     f"Len: {avg_len:.1f}px | Thk: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_str, fontsize=10, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    # Target dark/grey asphalt pixels specifically (ignoring bright lane lines >140 and bushes)
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    # Strict geometric roadway window (keeps middle highway region, zeroes out side bushes completely)
    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(road_roi_mask, (int(w * 0.12), 0), (int(w * 0.88), h), 255, -1)

    # Final binary road mask: Road pixels are 255 (1), everything else is strictly 0
    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)

    # Column 2 display: Show original colors ONLY inside the road mask (background set to absolute zero/black)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: HIGH-CONTRAST LAPLACIAN EDGE DETECTION FOR CRACKS ONLY
    # ==============================================================================
    # Enhance subtle crack contrast using CLAHE
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)

    # Gentle blur to reduce rough asphalt grain while preserving crack edges
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian second-derivative filter
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold to isolate sharp crack edges
    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Restrict edge map strictly inside the binary road mask (eliminating lane line edges and side bushes)
    laplacian_binary_cracks = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=binary_road_mask)

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    # Column 1: Original UAV Image
    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    # Column 2: Road Region Only (Background & Bushes strictly 0 / Black)
    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    # Column 3: Clear Pavement Cracks Edge Map
    axes_prep[2].imshow(laplacian_binary_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cracks Clear)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 15

    lines_polar = cv2.HoughLines(laplacian_binary_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]

        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw the line overlay
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            # Mask line to isolate intersecting crack pixels
            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]] # x, y coordinates
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            # Estimate Width and Thickness from pixel area distribution along the line segment
            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width # Equivalent mean pixel thickness representation

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_binary_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES WITH METRICS DISPLAY
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"ROAD-ISOLATED HOUGH RANKINGS & CRACK METRICS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# 1. UPLOAD UAV DATASET IMAGE FROM LOCAL FOLDER
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_img.shape[:2]

    # Convert image to HSV color space
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ==============================================================================
    # STAGE 0: BINARY ROAD MASK (ROAD = 1/255, BACKGROUND/BUSHES = 0)
    # ==============================================================================
    lower_dark_grey_asphalt = np.array([0, 0, 40])
    upper_dark_grey_asphalt = np.array([180, 40, 140])
    asphalt_color_mask = cv2.inRange(hsv, lower_dark_grey_asphalt, upper_dark_grey_asphalt)

    road_roi_mask = np.zeros((h, w), dtype=np.uint8)
    # Narrow the horizontal window slightly to exclude hard outer road edges/curbs
    cv2.rectangle(road_roi_mask, (int(w * 0.18), 0), (int(w * 0.82), h), 255, -1)

    binary_road_mask = cv2.bitwise_and(asphalt_color_mask, road_roi_mask)
    img_road_only_display = cv2.bitwise_and(img_rgb, img_rgb, mask=binary_road_mask)

    # ==============================================================================
    # STEP 1: HIGH-CONTRAST LAPLACIAN & OBJECT/CORNER FILTERING
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 22, 255, cv2.THRESH_BINARY)

    # Restrict strictly inside road mask
    road_edges = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=binary_road_mask)

    # Remove cars and long straight boundary/lane lines using morphological filtering & contour analysis
    # Remove large blobs (like cars/vehicles) and long continuous straight lines
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(road_edges, connectivity=8)

    filtered_cracks = np.zeros_like(road_edges)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]

        # Filter criteria: Cracks are thin and have moderate length (not huge cars or infinite boundary lines)
        max_dim = max(width, height)
        min_dim = min(width, height)

        # Keep components that look like thin crack segments
        if 5 < area < 800 and min_dim < 6 and max_dim < 250:
            filtered_cracks[labels == i] = 255

    # ------------------------------------------------------------------------------
    # DISPLAY STAGE 0 & STEP 1 PREPROCESSING STAGES
    # ------------------------------------------------------------------------------
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & FILTERED CRACK EDGE DETECTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original UAV Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(img_road_only_display)
    axes_prep[1].set_title("2. Road Region Only (Bushes Zeroed)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(filtered_cracks, cmap='gray')
    axes_prep[2].set_title("3. Filtered Cracks Only (Cars & Edges Removed)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # STEP 2: POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(filtered_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, filtered_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # ==============================================================================
    # DISPLAY ALL 5 RANKED INTERSECTION SLICE IMAGES WITH METRICS DISPLAY
    # ==============================================================================
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"CLEANED CRACK HOUGH RANKINGS & METRICS ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# COMBINED PIPELINE: SATURATION MASK -> LAPLACIAN -> HOUGH TRANSFORM & METRICS
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Convert BGR to HSV color space & extract Saturation (S) component
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)

    # 3. Threshold and invert saturation binary mask
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 4. Apply bitwise_and between inverted mask and original RGB image
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 5. LAPLACIAN EDGE DETECTION ON BITWISE AND OUTPUT
    # ==============================================================================
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Clean edge map using contour filtering to remove large objects/lines
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(laplacian_binary, connectivity=8)
    filtered_cracks = np.zeros_like(laplacian_binary)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        max_dim = max(width, height)
        min_dim = min(width, height)
        if 5 < area < 800 and min_dim < 6 and max_dim < 250:
            filtered_cracks[labels == i] = 255

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"PIPELINE: MASKING & LAPLACIAN EDGES ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(filtered_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Cracks)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 6. POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(filtered_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, filtered_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS & CRACK METRICS ON BITWISE OUTPUT ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: BITWISE AND OUTPUT AS ROAD MASK -> LAPLACIAN -> HOUGH & METRICS
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Column 2: Bitwise AND Output (Road only, background/bushes blacked out)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # Create a single-channel binary road mask directly from the inverted saturation mask
    road_mask_from_bitwise = s_binary_inv

    # ==============================================================================
    # 4. LAPLACIAN EDGE DETECTION CONFINED STRICTLY TO THE BITWISE ROAD MASK
    # ==============================================================================
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_img)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # RESTRICT LAPLACIAN EDGES STRICTLY TO THE BITWISE AND ROAD MASK
    road_constrained_edges = cv2.bitwise_and(laplacian_binary, laplacian_binary, mask=road_mask_from_bitwise)

    # Clean edge map using component filtering to remove vehicles and large borders
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(road_constrained_edges, connectivity=8)
    filtered_cracks = np.zeros_like(road_constrained_edges)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        max_dim = max(width, height)
        min_dim = min(width, height)
        if 5 < area < 800 and min_dim < 6 and max_dim < 250:
            filtered_cracks[labels == i] = 255

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"BITWISE MASKED ROAD & LAPLACIAN EDGES ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Mask)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(filtered_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Strictly Road-Masked)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(filtered_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, filtered_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON BITWISE-MASKED ROAD EDGES ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: APPLY LAPLACIAN DIRECTLY ON BITWISE AND OUTPUT IMAGE
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Column 2: Bitwise AND Output (Road only, background/bushes blacked out)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION DIRECTLY ON THE BITWISE AND OUTPUT IMAGE
    # ==============================================================================
    # Convert the color Bitwise AND output image to grayscale
    gray_from_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_from_bitwise)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Clean edge map using component filtering to remove vehicles and large borders
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(laplacian_binary, connectivity=8)
    filtered_cracks = np.zeros_like(laplacian_binary)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        max_dim = max(width, height)
        min_dim = min(width, height)
        if 10 < area:
            filtered_cracks[labels == i] = 255

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"LAPLACIAN DIRECTLY ON BITWISE OUTPUT ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(filtered_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (From Bitwise Output)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(filtered_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, filtered_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON BITWISE-PROCESSED EDGES ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: APPLY LAPLACIAN DIRECTLY ON BITWISE AND OUTPUT IMAGE
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Column 2: Bitwise AND Output (Road only, background/bushes blacked out)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION DIRECTLY ON THE BITWISE AND OUTPUT IMAGE
    # ==============================================================================
    # Convert the color Bitwise AND output image to grayscale
    gray_from_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_from_bitwise)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_binary = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Clean edge map using component filtering to remove vehicles and large borders
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(laplacian_binary, connectivity=8)

    # DISPLAY RAW CONNECTED COMPONENTS LABELS
    plt.figure(figsize=(6, 6))
    plt.imshow(labels, cmap='nipy_spectral')
    plt.title(f"Raw Connected Components (Total Blobs: {num_labels - 1})")
    plt.axis('off')
    plt.show()

    filtered_cracks = np.zeros_like(laplacian_binary)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        '''width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]
        max_dim = max(width, height)
        min_dim = min(width, height)'''
        if 10 < area:
            filtered_cracks[labels == i] = 255

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"LAPLACIAN DIRECTLY ON BITWISE OUTPUT ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(filtered_cracks, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (From Bitwise Output)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(filtered_cracks, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, filtered_cracks, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON BITWISE-PROCESSED EDGES ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# STANDALONE SCRIPT: SATURATION BINARY, MORPHOLOGY, & BITWISE AND WITH RGB
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Convert BGR to HSV color space
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # 3. Extract Saturation (S) component
    _, s_channel, _ = cv2.split(hsv_img)

    # 4. Convert Saturation component into a binary image via thresholding
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # 5. Inverse the binary mask
    s_binary_inv = cv2.bitwise_not(s_binary)

    # ==============================================================================
    # 6. APPLY MORPHOLOGICAL OPERATIONS (OPENING & CLOSING)
    # ==============================================================================
    kernel = np.ones((5, 5), np.uint8)

    # Morphology on Saturation Binary
    s_binary_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_binary_close = cv2.morphologyEx(s_binary, cv2.MORPH_CLOSE, kernel)

    # Morphology on Inverted Saturation Binary
    s_binary_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_binary_inv_close = cv2.morphologyEx(s_binary_inv, cv2.MORPH_CLOSE, kernel)

    # 7. Apply bitwise_and using the processed inverted mask
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv_close)

    # 8. Plot and display results side-by-side
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f"Morphological Operations on Masks ({filename})", fontsize=16, weight='bold')

    # Row 1: Saturation Binary pipeline
    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title("Original RGB")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(s_binary, cmap='gray')
    axes[0, 1].set_title("Saturation Binary (Before)")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(s_binary_open, cmap='gray')
    axes[0, 2].set_title("Sat Binary: Morph Open")
    axes[0, 2].axis('off')

    axes[0, 3].imshow(s_binary_close, cmap='gray')
    axes[0, 3].set_title("Sat Binary: Morph Close")
    axes[0, 3].axis('off')

    # Row 2: Inverted Saturation Binary pipeline
    axes[1, 0].imshow(rgb_masked_output)
    axes[1, 0].set_title("Bitwise AND Output")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(s_binary_inv, cmap='gray')
    axes[1, 1].set_title("Inverted Binary (Before)")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(s_binary_inv_open, cmap='gray')
    axes[1, 2].set_title("Inv Binary: Morph Open")
    axes[1, 2].axis('off')

    axes[1, 3].imshow(s_binary_inv_close, cmap='gray')
    axes[1, 3].set_title("Inv Binary: Morph Close")
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: LAPLACIAN DIRECTLY ON BITWISE AND OUTPUT (ROADWAY ONLY)
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Bitwise AND Output: Road region is retained in color, background is black
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION DIRECTLY ON THE BITWISE AND OUTPUT IMAGE
    # ==============================================================================
    # Convert the colored Bitwise AND output image directly to grayscale
    gray_masked_output = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Enhance contrast and apply Gaussian blur to smooth noise
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked_output)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges across the entire bitwise output image
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold to create the binary edge map (no extra crop masks applied)
    _, laplacian_edge_map = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & LAPLACIAN EDGE MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Only)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_edge_map, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Roadway Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS (LENGTH, WIDTH, THICKNESS)
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(laplacian_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 20),
        (20, 40),
        (40, 60),
        (60, 80),
        (80, 100)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_edge_map, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON ROADWAY EDGE MAP ({filename})", fontsize=16, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: MIDDLE & LOWER RANK HOUGH TRANSFORM FOCUS (EXCLUDING HIGHER RANKS)
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Bitwise AND Output: Road region is retained in color, background is black
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION DIRECTLY ON THE BITWISE AND OUTPUT IMAGE
    # ==============================================================================
    gray_masked_output = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked_output)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_edge_map = cv2.threshold(laplacian_norm, 20, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 3, figsize=(18, 5))
    fig_prep.suptitle(f"ISOLATED ROADWAY & LAPLACIAN EDGE MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Only)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_edge_map, cmap='gray')
    axes_prep[2].set_title("3. Laplacian Edge Map (Roadway Only)")
    axes_prep[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & FOCUS ON MIDDLE/LOWER RANKS ONLY
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 12

    lines_polar = cv2.HoughLines(laplacian_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(0, 0, 255)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # Adjusted ranges focusing ONLY on Middle and Lower ranks (ignoring top ranks 1-40)
    middle_lower_ranges = [
        (41, 60),   # Middle-High
        (61, 80),   # Middle
        (81, 100),  # Lower-Middle
        (101, 125), # Lower
        (125, 150)  # Deep Lower
    ]

    slice_results = []
    for start, end in middle_lower_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_edge_map, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    # Display Middle & Lower Ranked Hough Transformations with Metrics
    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"MIDDLE & LOWER RANK HOUGH EDGES (EXCLUDING HIGHER RANKS - {filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: LAPLACIAN INTENSITY SEGMENTATION (MIDDLE & LOWER MAGNITUDES ONLY)
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Bitwise AND Output: Road region retained in color, background blacked out
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. LAPLACIAN INTENSITY DECOMPOSITION (DISCARDING HIGHER STRENGTHS)
    # ==============================================================================
    gray_masked_output = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked_output)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Segment Laplacian magnitudes into Lower, Middle, and Higher intensity bands:
    # - Lower Magnitude Edges: pixel values between 15 and 45
    # - Middle Magnitude Edges: pixel values between 45 and 90
    # - Higher Magnitude Edges: pixel values > 90 (EXCLUDED)

    _, lower_lap_mask = cv2.threshold(laplacian_norm, 15, 255, cv2.THRESH_BINARY)
    _, upper_lower_mask = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    lower_edges = cv2.bitwise_and(lower_lap_mask, cv2.bitwise_not(upper_lower_mask))

    _, mid_lap_mask = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    _, upper_mid_mask = cv2.threshold(laplacian_norm, 90, 255, cv2.THRESH_BINARY)
    middle_edges = cv2.bitwise_and(mid_lap_mask, cv2.bitwise_not(upper_mid_mask))

    # Combine Middle and Lower edges only (ignoring Higher edges completely)
    middle_lower_combined_edges = cv2.bitwise_or(lower_edges, middle_edges)

    # Display Preprocessing & Laplacian Segmentation Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"LAPLACIAN INTENSITY SPLIT (MIDDLE & LOWER ONLY) - ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(lower_edges, cmap='gray')
    axes_prep[2].set_title("3. Lower Laplacian Edges")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(middle_lower_combined_edges, cmap='gray')
    axes_prep[3].set_title("4. Combined Middle + Lower Edges")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM ON MIDDLE & LOWER LAPLACIAN EDGES
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 10

    lines_polar = cv2.HoughLines(middle_lower_combined_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(0, 0, 255)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, middle_lower_combined_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH ON MIDDLE & LOWER LAPLACIAN INTENSITY EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: EXTRACTING SUBTLE ROADWAY CRACKS VIA LOWER & MIDDLE LAPLACIAN EDGES
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary mask for road
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # 3. Bitwise AND Output: Road region retained in color, background blacked out
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 4. ISOLATING LOWER & MIDDLE LAPLACIAN EDGES (EXCLUDING HIGHER INTENSITY EDGES)
    # ==============================================================================
    gray_masked_output = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Mild enhancement and blur to preserve fine crack details while reducing noise
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked_output)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Thresholding to separate intensities:
    # - Lower Range (Subtle cracks): Magnitude between 10 and 35
    # - Middle Range (Moderate cracks/textures): Magnitude between 35 and 70
    # - Higher Range (Lane lines & strong borders): > 70 (COMPLETELY EXCLUDED)

    _, lower_mask = cv2.threshold(laplacian_norm, 10, 255, cv2.THRESH_BINARY)
    _, cutoff_lower = cv2.threshold(laplacian_norm, 35, 255, cv2.THRESH_BINARY)
    subtle_lower_edges = cv2.bitwise_and(lower_mask, cv2.bitwise_not(cutoff_lower))

    _, mid_mask = cv2.threshold(laplacian_norm, 35, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(laplacian_norm, 70, 255, cv2.THRESH_BINARY)
    moderate_middle_edges = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    # Combine Lower and Middle edges exclusively for crack detection
    crack_focused_edges = cv2.bitwise_or(subtle_lower_edges, moderate_middle_edges)

    # Display Preprocessing Steps showing the Breakdown
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"LOWER & MIDDLE LAPLACIAN CRACK EXTRACTION ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Only)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(subtle_lower_edges, cmap='gray')
    axes_prep[2].set_title("3. Lower Laplacian Edges (Subtle)")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(crack_focused_edges, cmap='gray')
    axes_prep[3].set_title("4. Combined Lower + Middle (Crack Map)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS ON CRACK-FOCUSED EDGES
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(crack_focused_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, crack_focused_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON LOWER & MIDDLE CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: MIDDLE-RANK RGB BITWISE FILTERING -> LAPLACIAN EDGE DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. APPLY MIDDLE-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT (EXCLUDING HIGHER)
    # ==============================================================================
    # Convert RGB masked output to grayscale intensity to evaluate pixel ranks/magnitudes
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Isolate the middle-intensity range (excluding very dark pixels and very bright high-rank components)
    # Adjust lower and upper bounds as needed for your specific dataset lighting
    _, mid_lower = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    _, mid_upper = cv2.threshold(gray_bitwise, 160, 255, cv2.THRESH_BINARY)
    middle_ranked_bitwise = cv2.bitwise_and(mid_lower, cv2.bitwise_not(mid_upper))

    # Re-apply back to the color space or mask to isolate only middle-rank road features
    middle_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=middle_ranked_bitwise)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE MIDDLE-RANK BITWISE FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(middle_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 15, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"MIDDLE-RANK RGB BITWISE -> LAPLACIAN CRACK MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(middle_rgb_filtered)
    axes_prep[2].set_title("3. Middle-Rank Filtered RGB Bitwise")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Crack Focus)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 10

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON MIDDLE-RANKED CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: LOWER-RANK RGB BITWISE FILTERING -> LAPLACIAN CRACK DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. APPLY LOWER-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT (EXCLUDING HIGHER)
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Isolate lower-intensity pixels (capturing subtle variations/cracks, avoiding bright high-rank borders)
    _, lower_ranked_mask = cv2.threshold(gray_bitwise, 70, 255, cv2.THRESH_BINARY_INV)
    # Ensure we don't include absolute black background pixels
    _, non_black_mask = cv2.threshold(gray_bitwise, 15, 255, cv2.THRESH_BINARY)
    lower_ranked_bitwise_mask = cv2.bitwise_and(lower_ranked_mask, non_black_mask)

    # Apply lower-rank mask back to the RGB bitwise output
    lower_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=lower_ranked_bitwise_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE LOWER-RANK BITWISE FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(lower_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"LOWER-RANK RGB BITWISE -> LAPLACIAN CRACK MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(lower_rgb_filtered)
    axes_prep[2].set_title("3. Lower-Rank Filtered RGB Bitwise")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Lower Rank Only)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON LOWER-RANKED CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: COMBINED LOWER & MIDDLE-RANK RGB BITWISE -> LAPLACIAN CRACK DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. COMBINE LOWER & MIDDLE-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Define Lower-Rank and Middle-Rank thresholds (excluding Higher-Rank strong structures)
    # - Lower range: Intensity between 15 and 60
    # - Middle range: Intensity between 60 and 130
    # - Higher range: > 130 (completely ignored)

    _, lower_mask = cv2.threshold(gray_bitwise, 15, 255, cv2.THRESH_BINARY)
    _, cutoff_lower = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    subtle_lower_rank = cv2.bitwise_and(lower_mask, cv2.bitwise_not(cutoff_lower))

    _, mid_mask = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)
    moderate_mid_rank = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    # Combine Lower and Middle ranks together for the RGB bitwise filtering mask
    combined_lower_mid_mask = cv2.bitwise_or(subtle_lower_rank, moderate_mid_rank)

    # Apply combined mask back to the RGB bitwise output
    combined_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=combined_lower_mid_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE COMBINED FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(combined_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"COMBINED LOWER & MIDDLE-RANK RGB -> LAPLACIAN CRACK MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(combined_rgb_filtered)
    axes_prep[2].set_title("3. Combined Lower+Mid RGB Filtered")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Cracks Focus)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON COMBINED LOWER+MID CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: ROAD MASK -> LAPLACIAN -> COMBINED LOWER & MIDDLE LAPLACIAN RANKS
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. COMPUTE LAPLACIAN DIRECTLY ON ROAD GRAYSCALE IMAGE (NO RGB FILTERING)
    # ==============================================================================
    gray_road = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_road)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # ==============================================================================
    # 4. ISOLATE LOWER & MIDDLE LAPLACIAN INTENSITIES (EXCLUDING HIGHER-RANK)
    # ==============================================================================
    # - Lower Laplacian Range (Subtle cracks): 15 to 45
    # - Middle Laplacian Range (Moderate features/cracks): 45 to 95
    # - Higher Laplacian Range (Lane lines & strong edges): > 95 (Ignored)

    _, lower_mask = cv2.threshold(laplacian_norm, 15, 255, cv2.THRESH_BINARY)
    _, cutoff_lower = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    subtle_lower_laplacian = cv2.bitwise_and(lower_mask, cv2.bitwise_not(cutoff_lower))

    _, mid_mask = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(laplacian_norm, 95, 255, cv2.THRESH_BINARY)
    moderate_mid_laplacian = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    # Combine Lower and Middle Laplacian Edges exclusively
    combined_lower_mid_laplacian = cv2.bitwise_or(subtle_lower_laplacian, moderate_mid_laplacian)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"COMBINED LOWER & MIDDLE LAPLACIAN MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Only)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(laplacian_norm, cmap='gray')
    axes_prep[2].set_title("3. Full Normalized Laplacian")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(combined_lower_mid_laplacian, cmap='gray')
    axes_prep[3].set_title("4. Combined Lower + Mid Laplacian (Cracks)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(combined_lower_mid_laplacian, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, combined_lower_mid_laplacian, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON COMBINED LOWER+MID LAPLACIAN ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: ISOLATING CRACKS BY SUPPRESSING VERTICAL EDGES & LANE LINES
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 3. Compute Laplacian on the road region
    gray_road = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_road)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    # Combine lower and middle Laplacian intensities
    _, lower_mask = cv2.threshold(laplacian_norm, 15, 255, cv2.THRESH_BINARY)
    _, cutoff_lower = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    subtle_lower = cv2.bitwise_and(lower_mask, cv2.bitwise_not(cutoff_lower))

    _, mid_mask = cv2.threshold(laplacian_norm, 45, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(laplacian_norm, 95, 255, cv2.THRESH_BINARY)
    moderate_mid = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    combined_edges = cv2.bitwise_or(subtle_lower, moderate_mid)

    # ==============================================================================
    # 4. REMOVE VERTICAL EDGE & CENTER LANE LINES USING MORPHOLOGICAL KERNELS
    # ==============================================================================
    # Create a vertical structural kernel to isolate long vertical lines (lanes/edges)
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))
    detected_vertical_lines = cv2.morphologyEx(combined_edges, cv2.MORPH_OPEN, vertical_kernel, iterations=1)

    # Subtract the detected vertical lane lines from our combined edge map
    crack_focused_map = cv2.subtract(combined_edges, detected_vertical_lines)

    # Clean up small isolated noise using a tiny closing kernel
    clean_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    crack_focused_map = cv2.morphologyEx(crack_focused_map, cv2.MORPH_CLOSE, clean_kernel)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"PURE CRACK ISOLATION (NO LANES/EDGES) - ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(combined_edges, cmap='gray')
    axes_prep[1].set_title("2. Combined Lower+Mid Edges")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(detected_vertical_lines, cmap='gray')
    axes_prep[2].set_title("3. Isolated Vertical Lanes/Edges")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(crack_focused_map, cmap='gray')
    axes_prep[3].set_title("4. Pure Cracks Map (Lanes Removed)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS ON TRUE CRACK MAP
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 6

    lines_polar = cv2.HoughLines(crack_focused_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(0, 255, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, crack_focused_map, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON ISOLATED CRACKS ONLY ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: MIDDLE-RANK RGB BITWISE -> LAPLACIAN CRACK DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. APPLY MIDDLE-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Define Middle-Rank thresholds exclusively (ignoring lower noise and higher structures)
    # - Middle range: Intensity between 60 and 130
    _, mid_mask = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)
    middle_rank_mask = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    # Apply middle-rank mask back to the RGB bitwise output
    middle_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=middle_rank_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE MIDDLE-RANK FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(middle_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"MIDDLE-RANK RGB -> LAPLACIAN CRACK MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(middle_rgb_filtered)
    axes_prep[2].set_title("3. Middle-Rank RGB Filtered")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Cracks Focus)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON MIDDLE-RANK CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: COMBINED LOWER & MIDDLE-RANK RGB BITWISE -> LAPLACIAN CRACK DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. COMBINE LOWER & MIDDLE-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Define Lower-Rank and Middle-Rank thresholds (excluding Higher-Rank strong structures)
    # - Lower range: Intensity between 15 and 60
    # - Middle range: Intensity between 60 and 130
    # - Higher range: > 130 (completely ignored)

    _, lower_mask = cv2.threshold(gray_bitwise, 15, 255, cv2.THRESH_BINARY)
    _, cutoff_lower = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    subtle_lower_rank = cv2.bitwise_and(lower_mask, cv2.bitwise_not(cutoff_lower))

    _, mid_mask = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)
    moderate_mid_rank = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))

    # Combine Lower and Middle ranks together for the RGB bitwise filtering mask
    combined_lower_mid_mask = cv2.bitwise_or(subtle_lower_rank, moderate_mid_rank)

    # Apply combined mask back to the RGB bitwise output
    combined_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=combined_lower_mid_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE COMBINED FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(combined_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"COMBINED LOWER & MIDDLE-RANK RGB -> LAPLACIAN CRACK MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(combined_rgb_filtered)
    axes_prep[2].set_title("3. Combined Lower+Mid RGB Filtered")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Cracks Focus)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON COMBINED LOWER+MID CRACK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: ISOLATING TRANSVERSE CRACKS BY REMOVING VERTICAL LANES & EDGES
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 3. Middle-Rank Filtering (60 to 130)
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    _, mid_mask = cv2.threshold(gray_bitwise, 60, 255, cv2.THRESH_BINARY)
    _, cutoff_mid = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)
    middle_rank_mask = cv2.bitwise_and(mid_mask, cv2.bitwise_not(cutoff_mid))
    middle_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=middle_rank_mask)

    # 4. Laplacian Edge Detection
    gray_for_laplacian = cv2.cvtColor(middle_rgb_filtered, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # ==============================================================================
    # 5. DIRECTIONAL FILTERING: REMOVE VERTICAL LANES & KEEP ONLY CRACKS
    # ==============================================================================
    # Create a tall vertical structuring element to target long vertical lane lines
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 30))
    detected_vertical_lines = cv2.morphologyEx(laplacian_crack_edges, cv2.MORPH_OPEN, vertical_kernel)

    # Subtract the vertical lines from the edge map to leave only transverse cracks
    pure_cracks_map = cv2.subtract(laplacian_crack_edges, detected_vertical_lines)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"VERTICAL LINE SUPPRESSION (CRACKS FOCUS) - ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[1].set_title("2. Raw Laplacian Edges (With Lanes)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(detected_vertical_lines, cmap='gray')
    axes_prep[2].set_title("3. Isolated Vertical Lanes/Edges")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(pure_cracks_map, cmap='gray')
    axes_prep[3].set_title("4. Pure Cracks Map (Lanes Removed)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 6. POLAR HOUGH TRANSFORM & CRACK METRICS ON PURE CRACK MAP
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(pure_cracks_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, pure_cracks_map, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON PURE CRACK MAP ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: HIGHER-RANK RGB BITWISE -> LAPLACIAN CRACK DETECTION
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) # Wait, keep color for display, use BGR to RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Standard Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. APPLY HIGHER-RANK FILTERING DIRECTLY ON RGB BITWISE OUTPUT
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Define Higher-Rank thresholds exclusively (ignoring lower and middle intensities)
    # - Higher range: Intensity strictly above 130
    _, higher_mask = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)

    # Apply higher-rank mask back to the RGB bitwise output
    higher_rgb_filtered = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=higher_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION ON THE HIGHER-RANK FILTERED IMAGE
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(higher_rgb_filtered, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Compute Laplacian edges
    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_crack_edges = cv2.threshold(laplacian_norm, 12, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"HIGHER-RANK RGB -> LAPLACIAN EDGE MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(higher_rgb_filtered)
    axes_prep[2].set_title("3. Higher-Rank RGB Filtered (>130)")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_crack_edges, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (High Focus)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 8

    lines_polar = cv2.HoughLines(laplacian_crack_edges, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_crack_edges, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON HIGHER-RANK EDGES ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ==============================================================================
# PIPELINE: HIGHER-RANK FILTERING DIRECTLY ON BITWISE AND OUTPUT -> LAPLACIAN
# ==============================================================================
print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR color space
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR to RGB for correct plotting representation
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Extract Saturation (S) component & create inverted binary road mask
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Bitwise AND Output (Road in color, background black)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # ==============================================================================
    # 3. APPLY HIGHER-RANK FILTERING ON THE BITWISE AND OUTPUT
    # ==============================================================================
    gray_bitwise = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Keep only higher-rank pixel intensities (> 130)
    _, higher_mask = cv2.threshold(gray_bitwise, 130, 255, cv2.THRESH_BINARY)
    higher_rank_rgb = cv2.bitwise_and(rgb_masked_output, rgb_masked_output, mask=higher_mask)

    # ==============================================================================
    # 4. APPLY LAPLACIAN EDGE DETECTION DIRECTLY ON HIGHER-RANK BITWISE OUTPUT
    # ==============================================================================
    gray_for_laplacian = cv2.cvtColor(higher_rank_rgb, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_for_laplacian)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    laplacian_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian_abs = cv2.convertScaleAbs(laplacian_64f)
    laplacian_norm = cv2.normalize(laplacian_abs, None, 0, 255, cv2.NORM_MINMAX)

    _, laplacian_edge_map = cv2.threshold(laplacian_norm, 15, 255, cv2.THRESH_BINARY)

    # Display Preprocessing Steps
    fig_prep, axes_prep = plt.subplots(1, 4, figsize=(22, 5))
    fig_prep.suptitle(f"HIGHER-RANK BITWISE AND -> LAPLACIAN EDGE MAP ({filename})", fontsize=14, weight='bold')

    axes_prep[0].imshow(img_rgb)
    axes_prep[0].set_title("1. Original RGB Image")
    axes_prep[0].axis('off')

    axes_prep[1].imshow(rgb_masked_output)
    axes_prep[1].set_title("2. Bitwise AND Output (Road Only)")
    axes_prep[1].axis('off')

    axes_prep[2].imshow(higher_rank_rgb)
    axes_prep[2].set_title("3. Higher-Rank Bitwise Output (>130)")
    axes_prep[2].axis('off')

    axes_prep[3].imshow(laplacian_edge_map, cmap='gray')
    axes_prep[3].set_title("4. Laplacian Edge Map (Higher-Rank)")
    axes_prep[3].axis('off')

    plt.tight_layout()
    plt.show()

    # ==============================================================================
    # 5. POLAR HOUGH TRANSFORM & CRACK METRICS EVALUATION
    # ==============================================================================
    rho_res = 1
    theta_res = np.pi / 180
    hough_threshold = 10

    lines_polar = cv2.HoughLines(laplacian_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    rank_ranges = [
        (1, 15),
        (16, 30),
        (31, 45),
        (46, 60),
        (61, 75)
    ]

    slice_results = []
    for start, end in rank_ranges:
        img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
            img_rgb, laplacian_edge_map, lines_polar, start_rank=start, end_rank=end
        )
        slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

    fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
    fig_ranks.suptitle(f"HOUGH RANKINGS ON HIGHER-RANK ROADWAY EDGE MAP ({filename})", fontsize=15, weight='bold')

    for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
        axes_ranks[idx].imshow(img_out)
        title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                      f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                      f"Thick: {avg_thick:.1f}px")
        axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
        axes_ranks[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR and convert to RGB
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. HSV & Saturation Thresholding
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # 3. Morphological operations on Saturation Binary (Open -> Close -> Erode)
    kernel = np.ones((5, 5), np.uint8)
    s_morph = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_morph = cv2.morphologyEx(s_morph, cv2.MORPH_CLOSE, kernel)
    s_clean_erode = cv2.erode(s_morph, kernel, iterations=1)

    # Invert to isolate road and eliminate roadside clutter/dots
    s_binary_inv = cv2.bitwise_not(s_clean_erode)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 4. Edge Detection (Sobel & Laplacian)
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Sobel (X and Y combined)
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Combine edges for comprehensive crack detection
    combined_edges = cv2.bitwise_or(lap_binary, sobel_binary)

    # ==============================================================================
    # 5. CONNECTED COMPONENTS & REMOVING RECTANGULAR BOXES (NON-RECTANGULAR CRACKS)
    # =================================================0=============================
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(combined_edges, connectivity=8)

    components_colored = np.zeros((combined_edges.shape[0], combined_edges.shape[1], 3), dtype=np.uint8)
    filtered_non_rectangular_cracks = np.zeros_like(combined_edges)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        # Calculate bounding box rectangularity (Area / Bounding Box Area)
        bbox_area = w * h if (w * h) > 0 else 1
        rectangularity = area / float(bbox_area)

        # Assign random color for component visualization map
        color = np.random.randint(50, 255, size=3).tolist()
        components_colored[labels == i] = color

        # Filter out tiny noise AND highly rectangular structures (vehicles, boxes, borders)
        # Cracks typically have lower/irregular rectangularity and specific size profiles
        if area > 8 and rectangularity < 0.65 and max(w, h) < 300:
            filtered_non_rectangular_cracks[labels == i] = 255

    # ==============================================================================
    # 6. PLOTTING & DISPLAY RESULTS
    # ==============================================================================
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f"Advanced Road Edge & Non-Rectangular Crack Extraction ({filename})", fontsize=15, weight='bold')

    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title("1. Original RGB Image")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(s_clean_erode, cmap='gray')
    axes[0, 1].set_title("2. Saturation Mask (Open+Close+Erode)")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(combined_edges, cmap='gray')
    axes[0, 2].set_title("3. Combined Edges (Sobel + Laplacian)")
    axes[0, 2].axis('off')

    axes[1, 0].imshow(components_colored)
    axes[1, 0].set_title(f"4. Connected Components ({num_labels - 1} blobs)")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(combined_edges, cmap='gray')
    axes[1, 1].set_title("5. Without Connected Components (Raw Edges)")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(filtered_non_rectangular_cracks, cmap='gray')
    axes[1, 2].set_title("6. Filtered Non-Rectangular Cracks")
    axes[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR and convert to RGB
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. HSV & Saturation Thresholding
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # 3. Separate Morphological Operations
    kernel = np.ones((10,10), np.uint8)
    s_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_close = cv2.morphologyEx(s_open, cv2.MORPH_CLOSE, kernel)
    s_erode = cv2.erode(s_close, kernel, iterations=1)

    # Invert final eroded mask to isolate road and clean borders
    s_binary_inv = cv2.bitwise_not(s_erode)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 4. Separate Edge Detection (Sobel & Laplacian)
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian Edge Map
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Sobel Edge Map
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Combine edges
    combined_edges = cv2.bitwise_or(lap_binary, sobel_binary)

    # 5. Connected Components & Non-Rectangular Filtering
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(combined_edges, connectivity=8)

    components_colored = np.zeros((combined_edges.shape[0], combined_edges.shape[1], 3), dtype=np.uint8)
    filtered_non_rectangular_cracks = np.zeros_like(combined_edges)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        bbox_area = w * h if (w * h) > 0 else 1
        rectangularity = area / float(bbox_area)

        color = np.random.randint(50, 255, size=3).tolist()
        components_colored[labels == i] = color

        # Filter out rectangular boxes/vehicles and keep non-rectangular cracks
        if area > 8 and rectangularity < 0.65 and max(w, h) < 300:
            filtered_non_rectangular_cracks[labels == i] = 255

    # ==============================================================================
    # 6. PLOTTING - ROW 1: MORPHOLOGY BREAKDOWN
    # ==============================================================================
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f"Morphology & Edge Breakdown Pipeline ({filename})", fontsize=16, weight='bold')

    axes[0, 0].imshow(s_binary, cmap='gray')
    axes[0, 0].set_title("1. Saturation Binary (Original)")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(s_open, cmap='gray')
    axes[0, 1].set_title("2. Morphological Open")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(s_close, cmap='gray')
    axes[0, 2].set_title("3. Morphological Close")
    axes[0, 2].axis('off')

    axes[0, 3].imshow(s_erode, cmap='gray')
    axes[0, 3].set_title("4. Erode Output")
    axes[0, 3].axis('off')

    # ==============================================================================
    # PLOTTING - ROW 2: EDGES, COMPONENTS, & FINAL CRACKS
    # ==============================================================================
    axes[1, 0].imshow(sobel_binary, cmap='gray')
    axes[1, 0].set_title("5. Sobel Edges (Separate)")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(lap_binary, cmap='gray')
    axes[1, 1].set_title("6. Laplacian Edges (Separate)")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(components_colored)
    axes[1, 2].set_title(f"7. Connected Components ({num_labels - 1})")
    axes[1, 2].axis('off')

    axes[1, 3].imshow(filtered_non_rectangular_cracks, cmap='gray')
    axes[1, 3].set_title("8. Final Non-Rectangular Cracks")
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load image in BGR and convert to RGB
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. HSV & Saturation Thresholding
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # 3. Separate Morphological Operations
    kernel = np.ones((5, 5), np.uint8)
    s_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_close = cv2.morphologyEx(s_open, cv2.MORPH_CLOSE, kernel)
    s_erode = cv2.erode(s_close, kernel, iterations=1)

    # Invert final eroded mask to isolate road and clean borders
    s_binary_inv = cv2.bitwise_not(s_erode)
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_binary_inv)

    # 4. Separate Edge Detection (Sobel & Laplacian)
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian Edge Map
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Sobel Edge Map
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Combine edges
    combined_edges = cv2.bitwise_or(lap_binary, sobel_binary)

    # 5. Connected Components & Non-Rectangular Filtering
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(combined_edges, connectivity=8)

    components_colored = np.zeros((combined_edges.shape[0], combined_edges.shape[1], 3), dtype=np.uint8)
    filtered_non_rectangular_cracks = np.zeros_like(combined_edges)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        bbox_area = w * h if (w * h) > 0 else 1
        rectangularity = area / float(bbox_area)

        color = np.random.randint(50, 255, size=3).tolist()
        components_colored[labels == i] = color

        # Filter out rectangular boxes/vehicles and keep non-rectangular cracks
        if area > 8 and rectangularity < 0.65 and max(w, h) < 300:
            filtered_non_rectangular_cracks[labels == i] = 255

    # ==============================================================================
    # 6. PLOTTING - ROW 1: MORPHOLOGY BREAKDOWN
    # ==============================================================================
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f"Morphology & Edge Breakdown Pipeline ({filename})", fontsize=16, weight='bold')

    axes[0, 0].imshow(s_binary, cmap='gray')
    axes[0, 0].set_title("1. Saturation Binary (Original)")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(s_open, cmap='gray')
    axes[0, 1].set_title("2. Morphological Open")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(s_close, cmap='gray')
    axes[0, 2].set_title("3. Morphological Close")
    axes[0, 2].axis('off')

    axes[0, 3].imshow(s_erode, cmap='gray')
    axes[0, 3].set_title("4. Erode Output")
    axes[0, 3].axis('off')

    # ==============================================================================
    # PLOTTING - ROW 2: EDGES, COMPONENTS, & FINAL CRACKS
    # ==============================================================================
    axes[1, 0].imshow(sobel_binary, cmap='gray')
    axes[1, 0].set_title("5. Sobel Edges (Separate)")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(lap_binary, cmap='gray')
    axes[1, 1].set_title("6. Laplacian Edges (Separate)")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(components_colored)
    axes[1, 2].set_title(f"7. Connected Components ({num_labels - 1})")
    axes[1, 2].axis('off')

    axes[1, 3].imshow(filtered_non_rectangular_cracks, cmap='gray')
    axes[1, 3].set_title("8. Final Non-Rectangular Cracks")
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load Image
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. HSV & Saturation Thresholding
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # ==============================================================================
    # 3. MORPHOLOGY & NOISE REMOVAL FOR SATURATION BINARY
    # ==============================================================================
    kernel = np.ones((5, 5), np.uint8)

    # Saturation Binary Pipeline (Open -> Close -> Erode)
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Remove small specks/dots completely using Connected Components filtering
    num_l, lab, st, _ = cv2.connectedComponentsWithStats(s_bin_erode, connectivity=8)
    s_bin_clean = np.zeros_like(s_bin_erode)
    for i in range(1, num_l):
        if st[i, cv2.CC_STAT_AREA] > 500:
            s_bin_clean[lab == i] = 255

    # ==============================================================================
    # 4. MORPHOLOGY & NOISE REMOVAL FOR INVERTED SATURATION BINARY
    # ==============================================================================
    # Inverted Binary Pipeline (Open -> Close -> Erode)
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Remove small specks/dots on inverted mask
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Masked RGB output using cleaned inverted mask
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # ==============================================================================
    # 5. EDGE DETECTION (SOBEL & LAPLACIAN)
    # ==============================================================================
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Laplacian
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Sobel
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    combined_edges = cv2.bitwise_or(lap_binary, sobel_binary)

    # ==============================================================================
    # 6. CONNECTED COMPONENTS & NON-RECTANGULAR CRACK FILTERING
    # ==============================================================================
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(combined_edges, connectivity=8)

    components_colored = np.zeros((combined_edges.shape[0], combined_edges.shape[1], 3), dtype=np.uint8)
    filtered_cracks = np.zeros_like(combined_edges)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        bbox_area = w * h if (w * h) > 0 else 1
        rectangularity = area / float(bbox_area)

        color = np.random.randint(50, 255, size=3).tolist()
        components_colored[labels == i] = color

        if area > 8 and rectangularity < 0.65 and max(w, h) < 300:
            filtered_cracks[labels == i] = 255

    # ==============================================================================
    # 7. DISPLAY SIDE-BY-SIDE IN MULTI-COLUMN ROWS
    # ==============================================================================

    # ROW 1: Inputs & Bitwise Masked Road Output
    fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
    fig1.suptitle(f"1. Base Image & Road Region Extraction ({filename})", fontsize=16, weight='bold')
    axes1[0].imshow(img_rgb); axes1[0].set_title("Original RGB"); axes1[0].axis('off')
    axes1[1].imshow(s_binary, cmap='gray'); axes1[1].set_title("Raw Saturation Binary"); axes1[1].axis('off')
    axes1[2].imshow(rgb_masked_output); axes1[2].set_title("Bitwise AND Road Output"); axes1[2].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 2: Saturation Binary Morphology Pipeline
    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle("2. Saturation Binary Morphological Processing", fontsize=16, weight='bold')
    axes2[0].imshow(s_binary, cmap='gray'); axes2[0].set_title("1. Original Binary"); axes2[0].axis('off')
    axes2[1].imshow(s_bin_open, cmap='gray'); axes2[1].set_title("2. Morph Open"); axes2[1].axis('off')
    axes2[2].imshow(s_bin_close, cmap='gray'); axes2[2].set_title("3. Morph Close"); axes2[2].axis('off')
    axes2[3].imshow(s_bin_clean, cmap='gray'); axes2[3].set_title("4. Cleaned Mask (No Dots)"); axes2[3].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 3: Inverted Saturation Binary Morphology Pipeline
    fig3, axes3 = plt.subplots(1, 4, figsize=(20, 5))
    fig3.suptitle("3. Inverted Saturation Binary Morphological Processing", fontsize=16, weight='bold')
    axes3[0].imshow(s_binary_inv, cmap='gray'); axes3[0].set_title("1. Inverted Binary"); axes3[0].axis('off')
    axes3[1].imshow(s_inv_open, cmap='gray'); axes3[1].set_title("2. Morph Open"); axes3[1].axis('off')
    axes3[2].imshow(s_inv_close, cmap='gray'); axes3[2].set_title("3. Morph Close"); axes3[2].axis('off')
    axes3[3].imshow(s_inv_clean, cmap='gray'); axes3[3].set_title("4. Cleaned Mask (No Dots)"); axes3[3].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 4: Edge Detection Outputs Side-by-Side
    fig4, axes4 = plt.subplots(1, 3, figsize=(18, 5))
    fig4.suptitle("4. Edge Detection Comparison", fontsize=16, weight='bold')
    axes4[0].imshow(sobel_binary, cmap='gray'); axes4[0].set_title("Sobel Edges"); axes4[0].axis('off')
    axes4[1].imshow(lap_binary, cmap='gray'); axes4[1].set_title("Laplacian Edges"); axes4[1].axis('off')
    axes4[2].imshow(combined_edges, cmap='gray'); axes4[2].set_title("Combined Edges (Sobel + Laplacian)"); axes4[2].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 5: Connected Components & Crack Extraction Side-by-Side
    fig5, axes5 = plt.subplots(1, 3, figsize=(18, 5))
    fig5.suptitle("5. Connected Components & Final Crack Detection", fontsize=16, weight='bold')
    axes5[0].imshow(combined_edges, cmap='gray'); axes5[0].set_title("Without Connected Components Filter"); axes5[0].axis('off')
    axes5[1].imshow(components_colored); axes5[1].set_title(f"Connected Components ({num_labels - 1} Blobs)"); axes5[1].axis('off')
    axes5[2].imshow(filtered_cracks, cmap='gray'); axes5[2].set_title("Final Non-Rectangular Cracks"); axes5[2].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload UAV dataset image from your local folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Load Image
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. HSV & Saturation Thresholding
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # ==============================================================================
    # 3. MORPHOLOGY & CLEANING FOR SATURATION BINARY
    # ==============================================================================
    kernel = np.ones((6, 6), np.uint8)

    # Saturation Binary Pipeline
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Remove small specks/dots completely
    num_l, lab, st, _ = cv2.connectedComponentsWithStats(s_bin_erode, connectivity=8)
    s_bin_clean = np.zeros_like(s_bin_erode)
    for i in range(1, num_l):
        if st[i, cv2.CC_STAT_AREA] > 500:
            s_bin_clean[lab == i] = 255

    # ==============================================================================
    # 4. MORPHOLOGY & CLEANING FOR INVERTED SATURATION BINARY
    # ==============================================================================
    # Inverted Binary Pipeline
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Remove small specks/dots on inverted mask
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Masked RGB output
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # ==============================================================================
    # 5. SEPARATE EDGE DETECTION (SOBEL & LAPLACIAN)
    # ==============================================================================
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Sobel Edge Detection (Separate)
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Laplacian Edge Detection (Separate)
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # ==============================================================================
    # 6. CONNECTED COMPONENTS & BORDER REMOVAL (SOBEL & LAPLACIAN SEPARATELY)
    # ==============================================================================
    def clean_edges_and_get_cc(edge_img):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(edge_img, connectivity=8)

        cc_colored = np.zeros((edge_img.shape[0], edge_img.shape[1], 3), dtype=np.uint8)
        clean_edges = np.zeros_like(edge_img)

        # Find maximum component area (which corresponds to the big outer border/frame)
        max_area = 0
        if num_labels > 1:
            max_area = max([stats[i, cv2.CC_STAT_AREA] for i in range(1, num_labels)])

        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]

            # Color component
            color = np.random.randint(50, 255, size=3).tolist()
            cc_colored[labels == i] = color

            # Filter out tiny noise specks (< 8 px) and the huge outer frame border
            if area > 8:
                clean_edges[labels == i] = 255

        return num_labels - 1, cc_colored, clean_edges

    sobel_num_blobs, sobel_cc_colored, sobel_cleaned = clean_edges_and_get_cc(sobel_binary)
    lap_num_blobs, lap_cc_colored, lap_cleaned = clean_edges_and_get_cc(lap_binary)

    # ==============================================================================
    # 7. DISPLAY SIDE-BY-SIDE PLOTS
    # ==============================================================================

    # ROW 1: Inputs & Masked Road
    fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
    fig1.suptitle(f"1. Base Image & Road Region Extraction ({filename})", fontsize=16, weight='bold')
    axes1[0].imshow(img_rgb); axes1[0].set_title("Original RGB"); axes1[0].axis('off')
    axes1[1].imshow(s_binary, cmap='gray'); axes1[1].set_title("Raw Saturation Binary"); axes1[1].axis('off')
    axes1[2].imshow(rgb_masked_output); axes1[2].set_title("Bitwise AND Road Output"); axes1[2].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 2: Saturation Binary Morphology
    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle("2. Saturation Binary Morphological Processing", fontsize=16, weight='bold')
    axes2[0].imshow(s_binary, cmap='gray'); axes2[0].set_title("1. Original Binary"); axes2[0].axis('off')
    axes2[1].imshow(s_bin_open, cmap='gray'); axes2[1].set_title("2. Morph Open"); axes2[1].axis('off')
    axes2[2].imshow(s_bin_close, cmap='gray'); axes2[2].set_title("3. Morph Close"); axes2[2].axis('off')
    axes2[3].imshow(s_bin_clean, cmap='gray'); axes2[3].set_title("4. Cleaned Mask (No Dots)"); axes2[3].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 3: Inverted Saturation Binary Morphology
    fig3, axes3 = plt.subplots(1, 4, figsize=(20, 5))
    fig3.suptitle("3. Inverted Saturation Binary Morphological Processing", fontsize=16, weight='bold')
    axes3[0].imshow(s_binary_inv, cmap='gray'); axes3[0].set_title("1. Inverted Binary"); axes3[0].axis('off')
    axes3[1].imshow(s_inv_open, cmap='gray'); axes3[1].set_title("2. Morph Open"); axes3[1].axis('off')
    axes3[2].imshow(s_inv_close, cmap='gray'); axes3[2].set_title("3. Morph Close"); axes3[2].axis('off')
    axes3[3].imshow(s_inv_clean, cmap='gray'); axes3[3].set_title("4. Cleaned Mask (No Dots)"); axes3[3].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 4: Separate Sobel Processing
    fig4, axes4 = plt.subplots(1, 3, figsize=(18, 5))
    fig4.suptitle("4. Sobel Edge Processing Pipeline", fontsize=16, weight='bold')
    axes4[0].imshow(sobel_binary, cmap='gray'); axes4[0].set_title("1. Raw Sobel Edges"); axes4[0].axis('off')
    axes4[1].imshow(sobel_cc_colored); axes4[1].set_title(f"2. Connected Components ({sobel_num_blobs} Blobs)"); axes4[1].axis('off')
    axes4[2].imshow(sobel_cleaned, cmap='gray'); axes4[2].set_title("3. Sobel Edges (Border Removed)"); axes4[2].axis('off')
    plt.tight_layout(); plt.show()

    # ROW 5: Separate Laplacian Processing
    fig5, axes5 = plt.subplots(1, 3, figsize=(18, 5))
    fig5.suptitle("5. Laplacian Edge Processing Pipeline", fontsize=16, weight='bold')
    axes5[0].imshow(lap_binary, cmap='gray'); axes5[0].set_title("1. Raw Laplacian Edges"); axes5[0].axis('off')
    axes5[1].imshow(lap_cc_colored); axes5[1].set_title(f"2. Connected Components ({lap_num_blobs} Blobs)"); axes5[1].axis('off')
    axes5[2].imshow(lap_cleaned, cmap='gray'); axes5[2].set_title("3. Laplacian Edges (Border Removed)"); axes5[2].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: LOAD IMAGE & CONVERT TO HSV
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading RGB image, converting to HSV, and extracting Saturation channel...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (BOTH NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode) & Filtering Road Area...")

    kernel = np.ones((21, 21), np.uint8)

    # Normal Saturation Morphology (Open -> Close -> Erode)
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology (Open -> Close -> Erode)
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Road Area Filtering (removes small non-road specks)
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Bitwise AND to extract clean road output
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # ==============================================================================
    # STEP 3: APPLY SOBEL ON EXTRACTED ROAD + CONNECTED COMPONENTS
    # ==============================================================================
    print("[EXECUTION] Step 3: Preprocessing extracted road with CLAHE + Gaussian Blur, then running Sobel Detection...")

    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Sobel Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Sobel Connected Components
    sobel_num, sobel_labels, _, _ = cv2.connectedComponentsWithStats(sobel_binary, connectivity=8)
    sobel_cc_colored = np.zeros((sobel_binary.shape[0], sobel_binary.shape[1], 3), dtype=np.uint8)
    for i in range(1, sobel_num):
        sobel_cc_colored[sobel_labels == i] = np.random.randint(50, 255, size=3).tolist()

    # ==============================================================================
    # STEP 4: APPLY LAPLACIAN ON EXTRACTED ROAD + CONNECTED COMPONENTS
    # ==============================================================================
    print("[EXECUTION] Step 4: Running Laplacian Detection on the preprocessed road image...")

    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Laplacian Connected Components
    lap_num, lap_labels, _, _ = cv2.connectedComponentsWithStats(lap_binary, connectivity=8)
    lap_cc_colored = np.zeros((lap_binary.shape[0], lap_binary.shape[1], 3), dtype=np.uint8)
    for i in range(1, lap_num):
        lap_cc_colored[lap_labels == i] = np.random.randint(50, 255, size=3).tolist()

    # ==============================================================================
    # PLOTTING RESULTS IN STREAMLINED SEQUENTIAL ROWS
    # ==============================================================================

    # Plot Row 1: Base Extraction
    print("[OUTPUT GENERATION] Rendering Figure 1: Base Image & Bitwise Extraction Output...")
    fig1, ax1 = plt.subplots(1, 3, figsize=(18, 5))
    fig1.suptitle(f"1. Base Image & Bitwise Road Extraction ({filename})", fontsize=15, weight='bold')
    ax1[0].imshow(img_rgb); ax1[0].set_title("Original RGB"); ax1[0].axis('off')
    ax1[1].imshow(s_binary, cmap='gray'); ax1[1].set_title("Raw Saturation Binary"); ax1[1].axis('off')
    ax1[2].imshow(rgb_masked_output); ax1[2].set_title("Bitwise AND Extracted Road"); ax1[2].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 2: Saturation Morphological Pipeline
    print("[OUTPUT GENERATION] Rendering Figure 2: Saturation Morphological Pipeline Output...")
    fig2, ax2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle("2. Saturation Morphological Pipeline", fontsize=15, weight='bold')
    ax2[0].imshow(s_binary, cmap='gray'); ax2[0].set_title("1. Raw Binary"); ax2[0].axis('off')
    ax2[1].imshow(s_bin_open, cmap='gray'); ax2[1].set_title("2. Morph Open"); ax2[1].axis('off')
    ax2[2].imshow(s_bin_close, cmap='gray'); ax2[2].set_title("3. Morph Close"); ax2[2].axis('off')
    ax2[3].imshow(s_bin_erode, cmap='gray'); ax2[3].set_title("4. Erode"); ax2[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 3: Inverted Saturation Morphological Pipeline
    print("[OUTPUT GENERATION] Rendering Figure 3: Inverted Saturation Morphological Pipeline Output...")
    fig3, ax3 = plt.subplots(1, 4, figsize=(20, 5))
    fig3.suptitle("3. Inverted Saturation Morphological Pipeline", fontsize=15, weight='bold')
    ax3[0].imshow(s_binary_inv, cmap='gray'); ax3[0].set_title("1. Inverted Binary"); ax3[0].axis('off')
    ax3[1].imshow(s_inv_open, cmap='gray'); ax3[1].set_title("2. Morph Open"); ax3[1].axis('off')
    ax3[2].imshow(s_inv_close, cmap='gray'); ax3[2].set_title("3. Morph Close"); ax3[2].axis('off')
    ax3[3].imshow(s_inv_erode, cmap='gray'); ax3[3].set_title("4. Erode"); ax3[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 4: Sobel Processing directly on Road
    print("[OUTPUT GENERATION] Rendering Figure 4: Sobel Edges + Connected Components Output...")
    fig4, ax4 = plt.subplots(1, 2, figsize=(14, 5))
    fig4.suptitle("4. Sobel Applied on Road + Connected Components", fontsize=15, weight='bold')
    ax4[0].imshow(sobel_binary, cmap='gray'); ax4[0].set_title("Sobel Edges"); ax4[0].axis('off')
    ax4[1].imshow(sobel_cc_colored); ax4[1].set_title(f"Sobel Connected Components ({sobel_num - 1} Blobs)"); ax4[1].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 5: Laplacian Processing directly on Road
    print("[OUTPUT GENERATION] Rendering Figure 5: Laplacian Edges + Connected Components Output...")
    fig5, ax5 = plt.subplots(1, 2, figsize=(14, 5))
    fig5.suptitle("5. Laplacian Applied on Road + Connected Components", fontsize=15, weight='bold')
    ax5[0].imshow(lap_binary, cmap='gray'); ax5[0].set_title("Laplacian Edges"); ax5[0].axis('off')
    ax5[1].imshow(lap_cc_colored); ax5[1].set_title(f"Laplacian Connected Components ({lap_num - 1} Blobs)"); ax5[1].axis('off')
    plt.tight_layout(); plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: LOAD IMAGE & CONVERT TO HSV
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading RGB image, converting to HSV, and extracting Saturation channel...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (BOTH NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode)...")

    kernel = np.ones((6, 6), np.uint8)

    # Normal Saturation Morphology (Open -> Close -> Erode)
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology (Open -> Close -> Erode)
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Bitwise AND to extract clean road output using inverted mask
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_erode)

    # ==============================================================================
    # STEP 3: APPLY SOBEL ON EXTRACTED ROAD
    # ==============================================================================
    print("[EXECUTION] Step 3: Preprocessing extracted road with CLAHE + Gaussian Blur, then running Sobel Detection...")

    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Sobel Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # ==============================================================================
    # STEP 4: APPLY LAPLACIAN ON EXTRACTED ROAD
    # ==============================================================================
    print("[EXECUTION] Step 4: Running Laplacian Detection on the preprocessed road image...")

    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # ==============================================================================
    # PLOTTING RESULTS IN STREAMLINED SEQUENTIAL ROWS
    # ==============================================================================

    # Plot Row 1: Base Extraction
    print("[OUTPUT GENERATION] Rendering Figure 1: Base Image & Bitwise Extraction Output...")
    fig1, ax1 = plt.subplots(1, 3, figsize=(18, 5))
    fig1.suptitle(f"1. Base Image & Bitwise Road Extraction ({filename})", fontsize=15, weight='bold')
    ax1[0].imshow(img_rgb); ax1[0].set_title("Original RGB"); ax1[0].axis('off')
    ax1[1].imshow(s_binary, cmap='gray'); ax1[1].set_title("Raw Saturation Binary"); ax1[1].axis('off')
    ax1[2].imshow(rgb_masked_output); ax1[2].set_title("Bitwise AND Extracted Road"); ax1[2].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 2: Saturation Morphological Pipeline
    print("[OUTPUT GENERATION] Rendering Figure 2: Saturation Morphological Pipeline Output...")
    fig2, ax2 = plt.subplots(1, 4, figsize=(20, 5))
    fig2.suptitle("2. Saturation Morphological Pipeline", fontsize=15, weight='bold')
    ax2[0].imshow(s_binary, cmap='gray'); ax2[0].set_title("1. Raw Binary"); ax2[0].axis('off')
    ax2[1].imshow(s_bin_open, cmap='gray'); ax2[1].set_title("2. Morph Open"); ax2[1].axis('off')
    ax2[2].imshow(s_bin_close, cmap='gray'); ax2[2].set_title("3. Morph Close"); ax2[2].axis('off')
    ax2[3].imshow(s_bin_erode, cmap='gray'); ax2[3].set_title("4. Erode"); ax2[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 3: Inverted Saturation Morphological Pipeline
    print("[OUTPUT GENERATION] Rendering Figure 3: Inverted Saturation Morphological Pipeline Output...")
    fig3, ax3 = plt.subplots(1, 4, figsize=(20, 5))
    fig3.suptitle("3. Inverted Saturation Morphological Pipeline", fontsize=15, weight='bold')
    ax3[0].imshow(s_binary_inv, cmap='gray'); ax3[0].set_title("1. Inverted Binary"); ax3[0].axis('off')
    ax3[1].imshow(s_inv_open, cmap='gray'); ax3[1].set_title("2. Morph Open"); ax3[1].axis('off')
    ax3[2].imshow(s_inv_close, cmap='gray'); ax3[2].set_title("3. Morph Close"); ax3[2].axis('off')
    ax3[3].imshow(s_inv_erode, cmap='gray'); ax3[3].set_title("4. Erode"); ax3[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Row 4: Edge Detection Comparison (Sobel vs Laplacian) directly on extracted road
    print("[OUTPUT GENERATION] Rendering Figure 4: Sobel vs Laplacian Edge Detection Output...")
    fig4, ax4 = plt.subplots(1, 2, figsize=(14, 5))
    fig4.suptitle("4. Edge Detection Applied Directly on Extracted Road", fontsize=15, weight='bold')
    ax4[0].imshow(sobel_binary, cmap='gray'); ax4[0].set_title("Sobel Edges"); ax4[0].axis('off')
    ax4[1].imshow(lap_binary, cmap='gray'); ax4[1].set_title("Laplacian Edges"); ax4[1].axis('off')
    plt.tight_layout(); plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # Set kernel > 5 (7x7)

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 4: SOBEL EDGE MAP + CONNECTED COMPONENTS
    # ==============================================================================
    print("[EXECUTION] Step 4: Applying Sobel Edge Detection + Connected Components on Extracted Road...")

    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Sobel Connected Components
    sobel_num, sobel_labels, _, _ = cv2.connectedComponentsWithStats(sobel_binary, connectivity=8)
    sobel_cc_colored = np.zeros((sobel_binary.shape[0], sobel_binary.shape[1], 3), dtype=np.uint8)
    for i in range(1, sobel_num):
        sobel_cc_colored[sobel_labels == i] = np.random.randint(50, 255, size=3).tolist()

    # Plot Step 4: Sobel Edges + CC
    fig4, ax4 = plt.subplots(1, 2, figsize=(14, 5))
    fig4.suptitle("Step 4: Sobel Edge Map & Connected Components", fontsize=14, weight='bold')
    ax4[0].imshow(sobel_binary, cmap='gray'); ax4[0].set_title("Sobel Edge Map"); ax4[0].axis('off')
    ax4[1].imshow(sobel_cc_colored); ax4[1].set_title(f"Sobel Connected Components ({sobel_num - 1} Blobs)"); ax4[1].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 5: LAPLACIAN EDGE MAP + CONNECTED COMPONENTS
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Laplacian Edge Detection + Connected Components on Extracted Road...")

    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Laplacian Connected Components
    lap_num, lap_labels, _, _ = cv2.connectedComponentsWithStats(lap_binary, connectivity=8)
    lap_cc_colored = np.zeros((lap_binary.shape[0], lap_binary.shape[1], 3), dtype=np.uint8)
    for i in range(1, lap_num):
        lap_cc_colored[lap_labels == i] = np.random.randint(50, 255, size=3).tolist()

    # Plot Step 5: Laplacian Edges + CC
    fig5, ax5 = plt.subplots(1, 2, figsize=(14, 5))
    fig5.suptitle("Step 5: Laplacian Edge Map & Connected Components", fontsize=14, weight='bold')
    ax5[0].imshow(lap_binary, cmap='gray'); ax5[0].set_title("Laplacian Edge Map"); ax5[0].axis('off')
    ax5[1].imshow(lap_cc_colored); ax5[1].set_title(f"Laplacian Connected Components ({lap_num - 1} Blobs)"); ax5[1].axis('off')
    plt.tight_layout(); plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram, Edge Map, and CC
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy):
        # Calculate Gradient Angle (-180 to 180 / 0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Only consider edge pixels

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Plot 4 Subplots: Edge Map, Angle Map, Angle Histogram, Connected Components
        fig, ax = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
        ax[1, 0].set_title(f"3. {edge_name} Edge Angle Distribution (0° - 360°)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")

        # 4. Connected Components
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Sobel / Gradient Magnitude
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: SOBEL EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Sobel Operator...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Sobel Operator", sobel_binary, sobel_x, sobel_y)

    # ==============================================================================
    # STEP 5: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Gradient Magnitude...")
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y)

    # ==============================================================================
    # STEP 6: LAPLACIAN EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 6: Processing Laplacian Operator...")
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Deriving 1st-order gradients (dx, dy) from Laplacian output for orientation angle calculation
    lap_gx = cv2.Sobel(lap_norm, cv2.CV_64F, 1, 0, ksize=3)
    lap_gy = cv2.Sobel(lap_norm, cv2.CV_64F, 0, 1, ksize=3)
    process_and_plot_edge_detector("Laplacian Operator", lap_binary, lap_gx, lap_gy)

    # ==============================================================================
    # STEP 7: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 7: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram (with overlay values), Edge Map, and CC
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy):
        # Calculate Gradient Angle (0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute Statistical Metrics for display
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 4 Subplots: Edge Map, Angle Map with Text Overlay, Angle Histogram with Peak Text, CC
        fig, ax = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map + Values Overlay
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        # Overlay statistics box on Angle Map
        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram + Dominant Peak Text Overlay
        if len(valid_angles) > 0:
            counts, bins, patches = ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            # Find peak bin (dominant direction)
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            # Annotate peak on histogram
            ax[1, 0].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                              xy=(peak_angle_start + 5, peak_count),
                              xytext=(peak_angle_start + 25, peak_count * 0.85),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                              fontsize=10, weight='bold', color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

        ax[1, 0].set_title(f"3. {edge_name} Edge Angle Distribution (0° - 360°)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")
        ax[1, 0].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Sobel / Gradient Magnitude
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: SOBEL EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Sobel Operator...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Sobel Operator", sobel_binary, sobel_x, sobel_y)

    # ==============================================================================
    # STEP 5: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Gradient Magnitude...")
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y)

    # ==============================================================================
    # STEP 6: LAPLACIAN EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 6: Processing Laplacian Operator...")
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Deriving 1st-order gradients (dx, dy) from Laplacian output for orientation angle calculation
    lap_gx = cv2.Sobel(lap_norm, cv2.CV_64F, 1, 0, ksize=3)
    lap_gy = cv2.Sobel(lap_norm, cv2.CV_64F, 0, 1, ksize=3)
    process_and_plot_edge_detector("Laplacian Operator", lap_binary, lap_gx, lap_gy)

    # ==============================================================================
    # STEP 7: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 7: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

Original Image – Raw Saturation Binary - Bitwise AND Extracted Road – Saturation Binary And Inverted Saturation Binary – Morph(open/ Close), Erode – Sobel Edges, Gradient Angle, Histogram and  Connected  Components–Hough Transform (Sobel) Laplacian Sobel Edges​

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram (with overlay values), Edge Map, and CC
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy):
        # Calculate Gradient Angle (0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute Statistical Metrics for display
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 4 Subplots: Edge Map, Angle Map with Text Overlay, Angle Histogram with Peak Text, CC
        fig, ax = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map + Values Overlay
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        # Overlay statistics box on Angle Map
        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram + Dominant Peak Text Overlay
        if len(valid_angles) > 0:
            counts, bins, patches = ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            # Find peak bin (dominant direction)
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            # Annotate peak on histogram
            ax[1, 0].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                              xy=(peak_angle_start + 5, peak_count),
                              xytext=(peak_angle_start + 25, peak_count * 0.85),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                              fontsize=10, weight='bold', color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

        ax[1, 0].set_title(f"3. {edge_name} Edge Angle Distribution (0° - 360°)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")
        ax[1, 0].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        # Requested rank ranges: 10-20, 20-40, 40-60, 60-70, 70-100
        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"HOUGH RANKINGS ON {detector_name.upper()} EDGE MAP ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Sobel / Gradient Magnitude
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: SOBEL EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Sobel Operator...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Sobel Operator", sobel_binary, sobel_x, sobel_y)
    run_hough_rank_analysis("Sobel Operator", sobel_binary)

    # ==============================================================================
    # STEP 5: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Gradient Magnitude...")
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y)
    run_hough_rank_analysis("Gradient Magnitude", grad_binary)

    # ==============================================================================
    # STEP 6: LAPLACIAN EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 6: Processing Laplacian Operator...")
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Deriving 1st-order gradients (dx, dy) from Laplacian output for orientation angle calculation
    lap_gx = cv2.Sobel(lap_norm, cv2.CV_64F, 1, 0, ksize=3)
    lap_gy = cv2.Sobel(lap_norm, cv2.CV_64F, 0, 1, ksize=3)
    process_and_plot_edge_detector("Laplacian Operator", lap_binary, lap_gx, lap_gy)
    run_hough_rank_analysis("Laplacian Operator", lap_binary)

    # ==============================================================================
    # STEP 7: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 7: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y)
    run_hough_rank_analysis("Canny Detector", canny_edges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram (with overlay values), Edge Map, and CC
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy):
        # Calculate Gradient Angle (0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute Statistical Metrics for display
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 4 Subplots: Edge Map, Angle Map with Text Overlay, Angle Histogram with Peak Text, CC
        fig, ax = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map + Values Overlay
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        # Overlay statistics box on Angle Map
        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram + Dominant Peak Text Overlay
        if len(valid_angles) > 0:
            counts, bins, patches = ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            # Find peak bin (dominant direction)
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            # Annotate peak on histogram
            ax[1, 0].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                              xy=(peak_angle_start + 5, peak_count),
                              xytext=(peak_angle_start + 25, peak_count * 0.85),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                              fontsize=10, weight='bold', color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

        ax[1, 0].set_title(f"3. {edge_name} Edge Angle Distribution (0° - 360°)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")
        ax[1, 0].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        # Requested rank ranges: 10-20, 20-40, 40-60, 60-70, 70-100
        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"HOUGH RANKINGS ON {detector_name.upper()} EDGE MAP ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y)
    run_hough_rank_analysis("Gradient Magnitude", grad_binary)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y)
    run_hough_rank_analysis("Canny Detector", canny_edges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram, CC, and Highlighted Peak Edges
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        # Calculate Gradient Angle (0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute Statistical Metrics for display
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Create Highlight Map for Edges corresponding to Histogram Bars > threshold (default 3000)
        peak_highlight_map = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        # Background: grayscale version of the binary edge map
        peak_highlight_map[binary_edge_map > 0] = [80, 80, 80]

        if len(valid_angles) > 0:
            counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))

            # Identify bins where pixel count is greater than the threshold
            peak_bin_indices = np.where(counts > pixel_count_threshold)[0]

            # Palette of vibrant colors to highlight different high-peak angle ranges
            color_palette = [
                [255, 0, 0],     # Bright Red
                [0, 255, 255],   # Cyan
                [255, 255, 0],   # Yellow
                [255, 0, 255],   # Magenta
                [0, 255, 0],     # Lime Green
                [255, 128, 0]    # Orange
            ]

            highlighted_ranges = []
            for idx_enum, bin_idx in enumerate(peak_bin_indices):
                start_deg = bins[bin_idx]
                end_deg = bins[bin_idx + 1]
                count_val = counts[bin_idx]

                # Mask pixels belonging to this angle range and present in the binary edge map
                angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
                assigned_color = color_palette[idx_enum % len(color_palette)]

                peak_highlight_map[angle_mask] = assigned_color
                highlighted_ranges.append(f"{start_deg:.0f}°–{end_deg:.0f}° ({count_val} px)")

        # Plot 2x3 Grid: Edges, Angle Map, Histogram, CC, and High-Peak Edges Highlight
        fig, ax = plt.subplots(2, 3, figsize=(22, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram, CC & Peak Highlights)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[0, 2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            # Find single max peak bin for annotation
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            ax[0, 2].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                              xy=(peak_angle_start + 5, peak_count),
                              xytext=(peak_angle_start + 25, peak_count * 0.85),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                              fontsize=10, weight='bold', color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

            # Horizontal threshold reference line on histogram
            ax[0, 2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[0, 2].legend(loc='upper right')

        ax[0, 2].set_title(f"3. {edge_name} Angle Distribution (0° - 360°)", fontsize=12)
        ax[0, 2].set_xlabel("Angle (Degrees)")
        ax[0, 2].set_ylabel("Pixel Count")
        ax[0, 2].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 0].imshow(cc_colored)
        ax[1, 0].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 0].axis('off')

        # 5. Highlighted Peak Edges on Binary Image (> 3000 count)
        ax[1, 1].imshow(peak_highlight_map)
        ax[1, 1].set_title(f"5. Edges with Peak > {pixel_count_threshold} Pixels", fontsize=12, weight='bold')
        ax[1, 1].axis('off')

        # Text box listing highlighted peak ranges
        if highlighted_ranges:
            peaks_str = "High Peak Ranges (>3000):\n" + "\n".join(highlighted_ranges)
        else:
            peaks_str = f"No peaks found above {pixel_count_threshold} pixels"

        ax[1, 2].text(0.1, 0.5, peaks_str, transform=ax[1, 2].transAxes,
                      fontsize=12, weight='bold', color='black',
                      verticalalignment='center',
                      bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', alpha=0.9, edgecolor='darkorange'))
        ax[1, 2].axis('off')

        plt.tight_layout()
        plt.show()

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"HOUGH RANKINGS ON {detector_name.upper()} EDGE MAP ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Gradient Magnitude", grad_binary)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Canny Detector", canny_edges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute Angle Map and Histogram display
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 1x3 Grid (Edges, Angle Map, Histogram)
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle Map & Histogram)", fontsize=14, weight='bold')

        # 1. Binary Edge Map
        ax[0].imshow(binary_edge_map, cmap='gray')
        ax[0].set_title(f"1. {edge_name} Binary Edges", fontsize=11)
        ax[0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[1].imshow(angle_map, cmap='hsv')
        ax[1].set_title(f"2. Gradient Angle Map", fontsize=11)
        ax[1].axis('off')
        fig.colorbar(ang_img, ax=ax[1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[1].text(0.03, 0.95, stats_text, transform=ax[1].transAxes,
                   fontsize=10, weight='bold', color='black',
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            ax[2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[2].legend(loc='upper right')

        ax[2].set_title(f"3. Angle Distribution (0° - 360°)", fontsize=11)
        ax[2].set_xlabel("Angle (Degrees)")
        ax[2].set_ylabel("Pixel Count")
        ax[2].grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

    # Helper function for drawing polar lines and computing geometric metrics
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # Function to analyze Hough Transform per individual histogram bin (> 3000 pixels)
    def run_hough_per_histogram_bin(detector_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        print(f"\n=========================================================")
        print(f"RUNNING PER-BIN HOUGH TRANSFORM FOR {detector_name.upper()}")
        print(f"=========================================================")

        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) == 0:
            print("No valid angles found.")
            return

        # Compute 36 histogram bins (10 degrees each from 0 to 360)
        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
        peak_bin_indices = np.where(counts > pixel_count_threshold)[0]

        if len(peak_bin_indices) == 0:
            print(f"No bins exceeded the threshold of {pixel_count_threshold} pixels.")
            return

        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        # Loop through each individual histogram bin exceeding the threshold
        for bin_idx in peak_bin_indices:
            start_deg = bins[bin_idx]
            end_deg = bins[bin_idx + 1]
            pixel_count = counts[bin_idx]

            print(f"\n[BIN PROCESSING] Angle Range: {start_deg:.0f}°–{end_deg:.0f}° | Pixel Count: {pixel_count} px")

            # Create an isolated binary edge map containing ONLY edge pixels from this specific bin
            bin_edge_map = np.zeros_like(binary_edge_map)
            angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
            bin_edge_map[angle_mask] = 255

            # Apply Hough Transform exclusively on this bin's edge pixels
            lines_polar = cv2.HoughLines(bin_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

            if lines_polar is None or len(lines_polar) == 0:
                print(f"  -> No lines detected for bin {start_deg:.0f}°–{end_deg:.0f}°.")
                continue

            # Define rank slices for this specific bin
            rank_ranges = [
                (1, 5),
                (5, 10),
                (10, 20),
                (20, 30),
                (30, 50)
            ]

            slice_results = []
            for start, end in rank_ranges:
                img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                    img_rgb, bin_edge_map, lines_polar, start_rank=start, end_rank=end
                )
                slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

            # Display 5 rank subplots for this specific histogram bin
            fig_bin, axes_bin = plt.subplots(1, 5, figsize=(25, 5))
            fig_bin.suptitle(f"{detector_name} | Bin: {start_deg:.0f}°–{end_deg:.0f}° ({pixel_count} px)", fontsize=13, weight='bold')

            for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
                axes_bin[idx].imshow(img_out)
                title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                              f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                              f"Thick: {avg_thick:.1f}px")
                axes_bin[idx].set_title(title_text, fontsize=9, weight='bold')
                axes_bin[idx].axis('off')

            plt.tight_layout()
            plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_histogram_bin("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_histogram_bin("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")


histogram edge pixel each bin bar (high and low is 100 Hough threshold).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute Angle Map and Histogram display
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 1x3 Grid (Edges, Angle Map, Histogram)
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle Map & Histogram)", fontsize=14, weight='bold')

        # 1. Binary Edge Map
        ax[0].imshow(binary_edge_map, cmap='gray')
        ax[0].set_title(f"1. {edge_name} Binary Edges", fontsize=11)
        ax[0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[1].imshow(angle_map, cmap='hsv')
        ax[1].set_title(f"2. Gradient Angle Map", fontsize=11)
        ax[1].axis('off')
        fig.colorbar(ang_img, ax=ax[1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[1].text(0.03, 0.95, stats_text, transform=ax[1].transAxes,
                   fontsize=10, weight='bold', color='black',
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            ax[2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[2].legend(loc='upper right')

        ax[2].set_title(f"3. Angle Distribution (0° - 360°)", fontsize=11)
        ax[2].set_xlabel("Angle (Degrees)")
        ax[2].set_ylabel("Pixel Count")
        ax[2].grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

    # Helper function for drawing polar lines and computing geometric metrics
    def draw_polar_lines_with_metrics(base_img, binary_edges, lines, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1wha

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # Function to process EACH INDIVIDUAL histogram bar/bin one by one
    # Function to process EACH INDIVIDUAL histogram bar/bin with a stricter threshold
    def run_hough_per_individual_histogram_bin(detector_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        print(f"\n=========================================================")
        print(f"RUNNING PER-BAR (BIN-BY-BIN) HOUGH TRANSFORM FOR {detector_name.upper()}")
        print(f"=========================================================")

        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) == 0:
            print("No valid angles found.")
            return

        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
        valid_bin_indices = np.where(counts > 0)[0]

        rho_res = 1
        theta_res = np.pi / 180

        # INCREASED THRESHOLD: Stricter voting requirement to avoid overlapping clutter
        hough_threshold = 100

        for bin_idx in valid_bin_indices:
            start_deg = bins[bin_idx]
            end_deg = bins[bin_idx + 1]
            pixel_count = counts[bin_idx]

            if pixel_count >= pixel_count_threshold:
                density_type = f"High-Density (>= {pixel_count_threshold}px)"
                line_color = (255, 0, 0) # Red
            else:
                density_type = f"Low-Density (< {pixel_count_threshold}px)"
                line_color = (0, 255, 0) # Green

            print(f"\n[PROCESSING BAR] Angle: {start_deg:.0f}°–{end_deg:.0f}° | Pixel Count: {pixel_count} px | {density_type}")

            bin_edge_map = np.zeros_like(binary_edge_map)
            angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
            bin_edge_map[angle_mask] = 255

            lines_polar = cv2.HoughLines(bin_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

            if lines_polar is None or len(lines_polar) == 0:
                print(f"  -> No lines detected for this angle bar (threshold too strict or no dominant line).")
                continue

            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_lines_with_metrics(
                img_rgb, bin_edge_map, lines_polar, color=line_color
            )

            plt.figure(figsize=(5, 5))
            plt.imshow(img_out)
            title_text = (f"{detector_name}\n"
                          f"Bar: {start_deg:.0f}°–{end_deg:.0f}° ({pixel_count}px)\n"
                          f"Type: {density_type}\n"
                          f"Lines: {lines_drawn} | Len: {avg_len:.1f}px")
            plt.title(title_text, fontsize=9, weight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    # Compute base gradients
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

histogram edge pixel each bin bar (hough threshold adjustment high 100 low 500)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute Angle Map and Histogram display
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle Map & Histogram)", fontsize=14, weight='bold')

        # 1. Binary Edge Map
        ax[0].imshow(binary_edge_map, cmap='gray')
        ax[0].set_title(f"1. {edge_name} Binary Edges", fontsize=11)
        ax[0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[1].imshow(angle_map, cmap='hsv')
        ax[1].set_title(f"2. Gradient Angle Map", fontsize=11)
        ax[1].axis('off')
        fig.colorbar(ang_img, ax=ax[1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[1].text(0.03, 0.95, stats_text, transform=ax[1].transAxes,
                   fontsize=10, weight='bold', color='black',
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            ax[2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[2].legend(loc='upper right')

        ax[2].set_title(f"3. Angle Distribution (0° - 360°)", fontsize=11)
        ax[2].set_xlabel("Angle (Degrees)")
        ax[2].set_ylabel("Pixel Count")
        ax[2].grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

    # Helper function for drawing polar lines and computing geometric metrics
    def draw_polar_lines_with_metrics(base_img, binary_edges, lines, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # Function to process EACH INDIVIDUAL histogram bar/bin with requested Hough threshold
    def run_hough_per_individual_histogram_bin(detector_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        print(f"\n=========================================================")
        print(f"RUNNING PER-BAR (BIN-BY-BIN) HOUGH TRANSFORM FOR {detector_name.upper()}")
        print(f"=========================================================")

        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) == 0:
            print("No valid angles found.")
            return

        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
        valid_bin_indices = np.where(counts > 0)[0]

        rho_res = 1
        theta_res = np.pi / 180

        # Loop through EACH bar/bin individually
        for bin_idx in valid_bin_indices:
            start_deg = bins[bin_idx]
            end_deg = bins[bin_idx + 1]
            pixel_count = counts[bin_idx]

            # Hough Threshold configured as requested: 500 for low-density bins (< 3000px)
            if pixel_count >= pixel_count_threshold:
                density_type = f"High-Density (>= {pixel_count_threshold}px) - Structured Road Lines"
                line_color = (255, 0, 0) # Red for major road boundaries
                hough_threshold = 100
            else:
                density_type = f"Low-Density (< {pixel_count_threshold}px) - Potential Fine Cracks"
                line_color = (0, 255, 0) # Green for subtle/jagged features
                hough_threshold = 500    # Set to 500 as requested

            print(f"\n[PROCESSING BAR] Angle: {start_deg:.0f}°–{end_deg:.0f}° | Pixel Count: {pixel_count} px | Hough Threshold: {hough_threshold} | {density_type}")

            bin_edge_map = np.zeros_like(binary_edge_map)
            angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
            bin_edge_map[angle_mask] = 255

            lines_polar = cv2.HoughLines(bin_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

            if lines_polar is None or len(lines_polar) == 0:
                print(f"  -> [NOTE] No straight lines detected. Real-world cracks are irregular, jagged, and curved,")
                print(f"     scattering edge pixels across multiple angles without forming clean linear traces here.")
                continue

            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_lines_with_metrics(
                img_rgb, bin_edge_map, lines_polar, color=line_color
            )

            plt.figure(figsize=(5, 5))
            plt.imshow(img_out)
            title_text = (f"{detector_name}\n"
                          f"Bar: {start_deg:.0f}°–{end_deg:.0f}° ({pixel_count}px)\n"
                          f"Type: {density_type}\n"
                          f"Lines: {lines_drawn} | Len: {avg_len:.1f}px")
            plt.title(title_text, fontsize=9, weight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    # Compute base gradients
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

Histogram edge pixel each bin bar (Hough threshold adjustment high 100 low 30)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute Angle Map and Histogram display
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle Map & Histogram)", fontsize=14, weight='bold')

        # 1. Binary Edge Map
        ax[0].imshow(binary_edge_map, cmap='gray')
        ax[0].set_title(f"1. {edge_name} Binary Edges", fontsize=11)
        ax[0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[1].imshow(angle_map, cmap='hsv')
        ax[1].set_title(f"2. Gradient Angle Map", fontsize=11)
        ax[1].axis('off')
        fig.colorbar(ang_img, ax=ax[1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[1].text(0.03, 0.95, stats_text, transform=ax[1].transAxes,
                   fontsize=10, weight='bold', color='black',
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            ax[2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[2].legend(loc='upper right')

        ax[2].set_title(f"3. Angle Distribution (0° - 360°)", fontsize=11)
        ax[2].set_xlabel("Angle (Degrees)")
        ax[2].set_ylabel("Pixel Count")
        ax[2].grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

    # Helper function for drawing polar lines and computing geometric metrics
    def draw_polar_lines_with_metrics(base_img, binary_edges, lines, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # Function to process EACH INDIVIDUAL histogram bar/bin with adaptive thresholds for straight lines vs. jagged cracks (Virisal)
    def run_hough_per_individual_histogram_bin(detector_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        print(f"\n=========================================================")
        print(f"RUNNING PER-BAR (BIN-BY-BIN) HOUGH TRANSFORM FOR {detector_name.upper()}")
        print(f"=========================================================")

        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) == 0:
            print("No valid angles found.")
            return

        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
        valid_bin_indices = np.where(counts > 0)[0]

        rho_res = 1
        theta_res = np.pi / 180

        # Loop through EACH bar/bin individually
        for bin_idx in valid_bin_indices:
            start_deg = bins[bin_idx]
            end_deg = bins[bin_idx + 1]
            pixel_count = counts[bin_idx]

            # ADAPTIVE HOUGH CONFIGURATION:
            # - High-Density (>= 3000px): Structured straight road lines -> Strict Hough Threshold = 500 (Red Lines)
            # - Low-Density (< 3000px): Jagged cracks / "Virisal" -> Sensitive Hough Threshold = 30 (Green Lines)
            if pixel_count >= pixel_count_threshold:
                density_type = f"High-Density (>= {pixel_count_threshold}px) - Structured Road Lines"
                line_color = (255, 0, 0) # Red for major road boundaries
                hough_threshold = 100    # Strict threshold for straight lines
            else:
                density_type = f"Low-Density (< {pixel_count_threshold}px) - Jagged Cracks"
                line_color = (0, 255, 0) # Green for fine, irregular crack traces
                hough_threshold = 30     # Sensitive low threshold to capture jagged segments

            print(f"\n[PROCESSING BAR] Angle: {start_deg:.0f}°–{end_deg:.0f}° | Pixel Count: {pixel_count} px | Hough Threshold: {hough_threshold} | {density_type}")

            bin_edge_map = np.zeros_like(binary_edge_map)
            angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
            bin_edge_map[angle_mask] = 255

            lines_polar = cv2.HoughLines(bin_edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

            if lines_polar is None or len(lines_polar) == 0:
                print(f"  -> [NOTE] Insufficient continuous alignment for this angle segment.")
                continue

            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_lines_with_metrics(
                img_rgb, bin_edge_map, lines_polar, color=line_color
            )

            plt.figure(figsize=(5, 5))
            plt.imshow(img_out)
            title_text = (f"{detector_name}\n"
                          f"Bar: {start_deg:.0f}°–{end_deg:.0f}° ({pixel_count}px)\n"
                          f"Type: {density_type}\n"
                          f"Lines: {lines_drawn} | Len: {avg_len:.1f}px")
            plt.title(title_text, fontsize=9, weight='bold')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    # Compute base gradients
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_per_individual_histogram_bin("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Angle Map, Angle Histogram, CC, and Highlighted Peak Edges
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        # Calculate Gradient Angle (0 to 360 deg)
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute Statistical Metrics for display
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Create Highlight Map for Edges corresponding to Histogram Bars > threshold
        peak_highlight_map = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        peak_highlight_map[binary_edge_map > 0] = [80, 80, 80]

        highlighted_ranges = []
        peak_filtered_edges = np.zeros_like(binary_edge_map)

        if len(valid_angles) > 0:
            counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
            peak_bin_indices = np.where(counts > pixel_count_threshold)[0]

            color_palette = [
                [255, 0, 0],     # Bright Red
                [0, 255, 255],   # Cyan
                [255, 255, 0],   # Yellow
                [255, 0, 255],   # Magenta
                [0, 255, 0],     # Lime Green
                [255, 128, 0]    # Orange
            ]

            for idx_enum, bin_idx in enumerate(peak_bin_indices):
                start_deg = bins[bin_idx]
                end_deg = bins[bin_idx + 1]
                count_val = counts[bin_idx]

                angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
                assigned_color = color_palette[idx_enum % len(color_palette)]

                peak_highlight_map[angle_mask] = assigned_color
                peak_filtered_edges[angle_mask] = 255  # Retain only peak angle pixels
                highlighted_ranges.append(f"{start_deg:.0f}°–{end_deg:.0f}° ({count_val} px)")

        # Plot 2x3 Grid
        fig, ax = plt.subplots(2, 3, figsize=(22, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram, CC & Peak Highlights)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[0, 2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            ax[0, 2].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                            xy=(peak_angle_start + 5, peak_count),
                            xytext=(peak_angle_start + 25, peak_count * 0.85),
                            arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                            fontsize=10, weight='bold', color='darkred',
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

            ax[0, 2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[0, 2].legend(loc='upper right')

        ax[0, 2].set_title(f"3. {edge_name} Angle Distribution (0° - 360°)", fontsize=12)
        ax[0, 2].set_xlabel("Angle (Degrees)")
        ax[0, 2].set_ylabel("Pixel Count")
        ax[0, 2].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 0].imshow(cc_colored)
        ax[1, 0].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 0].axis('off')

        # 5. Highlighted Peak Edges on Binary Image (> 3000 count)
        ax[1, 1].imshow(peak_highlight_map)
        ax[1, 1].set_title(f"5. Edges with Peak > {pixel_count_threshold} Pixels", fontsize=12, weight='bold')
        ax[1, 1].axis('off')

        if highlighted_ranges:
            peaks_str = "High Peak Ranges (>3000):\n" + "\n".join(highlighted_ranges)
        else:
            peaks_str = f"No peaks found above {pixel_count_threshold} pixels"

        ax[1, 2].text(0.1, 0.5, peaks_str, transform=ax[1, 2].transAxes,
                      fontsize=12, weight='bold', color='black',
                      verticalalignment='center',
                      bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', alpha=0.9, edgecolor='darkorange'))
        ax[1, 2].axis('off')

        plt.tight_layout()
        plt.show()

        # Return the filtered edge map containing ONLY the edge pixels from high peak ranges
        return peak_filtered_edges

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name} on Peak-Angle Filtered Edges...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"PEAK-FILTERED HOUGH RANKINGS ON {detector_name.upper()} ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    # Extract peak-angle filtered edge map (>3000 pixels) from Gradient Magnitude analysis
    peak_filtered_grad_binary = process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)

    # Run Hough rank analysis using ONLY the peak-filtered edge pixels
    run_hough_rank_analysis("Gradient Magnitude", peak_filtered_grad_binary)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    # Extract peak-angle filtered edge map (>3000 pixels) from Canny analysis
    peak_filtered_canny = process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)

    # Run Hough rank analysis using ONLY the peak-filtered edge pixels
    run_hough_rank_analysis("Canny Detector", peak_filtered_canny)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute Angle Map, Histogram, and return filtered edges along with peak text descriptions
    def process_and_extract_peak_edges(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        peak_filtered_edges = np.zeros_like(binary_edge_map)
        highlighted_ranges = []

        if len(valid_angles) > 0:
            counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
            peak_bin_indices = np.where(counts > pixel_count_threshold)[0]

            for bin_idx in peak_bin_indices:
                start_deg = bins[bin_idx]
                end_deg = bins[bin_idx + 1]
                count_val = counts[bin_idx]

                angle_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)
                peak_filtered_edges[angle_mask] = 255
                highlighted_ranges.append(f"{start_deg:.0f}°–{end_deg:.0f}° ({count_val}px)")

        # Simplified 1x3 Grid Display (Edges, Angle Map, Histogram) - Removed CC & Highlight Map plots
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle Map & Histogram)", fontsize=14, weight='bold')

        # 1. Binary Edge Map
        ax[0].imshow(binary_edge_map, cmap='gray')
        ax[0].set_title(f"1. {edge_name} Binary Edges", fontsize=11)
        ax[0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[1].imshow(angle_map, cmap='hsv')
        ax[1].set_title(f"2. Gradient Angle Map", fontsize=11)
        ax[1].axis('off')
        fig.colorbar(ang_img, ax=ax[1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[1].text(0.03, 0.95, stats_text, transform=ax[1].transAxes,
                   fontsize=10, weight='bold', color='black',
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[2].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            ax[2].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[2].legend(loc='upper right')

        ax[2].set_title(f"3. Angle Distribution (0° - 360°)", fontsize=11)
        ax[2].set_xlabel("Angle (Degrees)")
        ax[2].set_ylabel("Pixel Count")
        ax[2].grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

        return peak_filtered_edges, highlighted_ranges

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map, peak_ranges):
        print(f"[EXECUTION] Running Peak-Filtered Hough Rank Analysis for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        peaks_title_str = " | ".join(peak_ranges[:3])  # Display top peak info in main title
        fig_ranks.suptitle(f"PEAK-FILTERED HOUGH RANKINGS ON {detector_name.upper()} | Peaks (>3000px): {peaks_title_str}", fontsize=12, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            # Embed rank metrics and state matching histogram bins/pixel counts directly into the subplots
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    peak_filtered_grad_binary, grad_peak_ranges = process_and_extract_peak_edges("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Gradient Magnitude", peak_filtered_grad_binary, grad_peak_ranges)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)

    peak_filtered_canny, canny_peak_ranges = process_and_extract_peak_edges("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Canny Detector", peak_filtered_canny, canny_peak_ranges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n[EXECUTION] Processing file: {filename}")

    # ==============================================================================
    # STEP 1: LOAD IMAGE & SATURATION ROAD EXTRACTION PIPELINE
    # ==============================================================================
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # HSV Saturation Masking for Roadway Isolation
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    kernel = np.ones((7, 7), np.uint8)
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # CLAHE & Gaussian Blur
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Gradient X, Y computation for angle calculation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    angle_map = cv2.phase(sobel_x, sobel_y, angleInDegrees=True)

    # Edge Detection Maps
    canny_edges = cv2.Canny(blurred, 50, 150)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    detectors = [("Canny Detector", canny_edges), ("Gradient Magnitude", grad_binary)]

    # ==============================================================================
    # STEP 2: BIN-BY-BIN HISTOGRAM PEAK SPATIAL MAPPING FUNCTION
    # ==============================================================================
    def analyze_histogram_bars_on_image(edge_name, binary_edge_map, pixel_threshold=3000):
        # Filter angle calculation strictly to active binary edge pixels
        valid_edge_pixels = binary_edge_map > 0
        valid_angles = angle_map[valid_edge_pixels]

        if len(valid_angles) == 0:
            print(f"No valid edges detected for {edge_name}.")
            return

        # 36 Bins (0° to 360°, 10° width per bar)
        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360))
        max_bin_idx = np.argmax(counts)
        high_peak_indices = np.where(counts >= pixel_threshold)[0]

        print(f"\n=========================================================")
        print(f"HISTOGRAM BAR ANALYSIS FOR: {edge_name}")
        print(f"=========================================================")
        print(f"Global Maximum Peak Bar: Bin {max_bin_idx} ({bins[max_bin_idx]:.0f}° - {bins[max_bin_idx+1]:.0f}°) with {counts[max_bin_idx]} pixels")
        print(f"Bars Exceeding Threshold ({pixel_threshold} px): {len(high_peak_indices)} bar(s)")

        # -------------------------------------------------------------------------
        # DISPLAY 1: HIGHEST PEAK BAR (180° - 190°) OVERLAY DIRECTLY ON ROADWAY IMAGE
        # -------------------------------------------------------------------------
        peak_start = bins[max_bin_idx]
        peak_end = bins[max_bin_idx + 1]
        peak_count = counts[max_bin_idx]

        # Mask corresponding strictly to the highest peak bar
        highest_peak_mask = (angle_map >= peak_start) & (angle_map < peak_end) & (binary_edge_map > 0)

        # Create Visual Overlays
        # 1. Overlay on Binary Edge Map
        peak_on_binary = cv2.cvtColor(binary_edge_map, cv2.COLOR_GRAY2RGB)
        peak_on_binary[highest_peak_mask] = [255, 0, 0]  # Highlighted in Bright Red

        # 2. Overlay on Original RGB Roadway Image
        road_overlay = img_rgb.copy()
        # Thicken the highlighted peak edge pixels for clear visual visibility on high-res photos
        thick_mask = cv2.dilate(highest_peak_mask.astype(np.uint8), np.ones((3, 3), np.uint8))
        road_overlay[thick_mask > 0] = [255, 255, 0]  # Bright Yellow overlay on original road

        fig_max, ax_max = plt.subplots(1, 3, figsize=(18, 6))
        fig_max.suptitle(f"{edge_name}: SPATIAL LOCATION OF HIGHEST HISTOGRAM PEAK ({peak_start:.0f}°-{peak_end:.0f}°)",
                         fontsize=14, weight='bold')

        # Subplot 1: Histogram showing the isolated bar
        n_counts, n_bins, patches = ax_max[0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
        patches[max_bin_idx].set_facecolor('red')
        patches[max_bin_idx].set_edgecolor('black')
        ax_max[0].axhline(y=pixel_threshold, color='crimson', linestyle='--', label=f'Threshold ({pixel_threshold} px)')
        ax_max[0].set_title(f"Histogram (Peak Bar Marked in Red)\nCount: {peak_count} px", fontsize=11, weight='bold')
        ax_max[0].set_xlabel("Angle (Degrees)")
        ax_max[0].set_ylabel("Pixel Count")
        ax_max[0].legend(loc='upper right')
        ax_max[0].grid(True, linestyle='--', alpha=0.5)

        # Subplot 2: Peak pixels on Binary Map
        ax_max[1].imshow(peak_on_binary)
        ax_max[1].set_title(f"Binary Edges of Highest Peak\nRange: {peak_start:.0f}° - {peak_end:.0f}°", fontsize=11, weight='bold')
        ax_max[1].axis('off')

        # Subplot 3: Peak pixels mapped directly onto Roadway Photo
        ax_max[2].imshow(road_overlay)
        ax_max[2].set_title(f"Roadway Photo Overlay (Yellow Lines)\nShows exact road feature causing peak", fontsize=11, weight='bold')
        ax_max[2].axis('off')

        plt.tight_layout()
        plt.show()

        # -------------------------------------------------------------------------
        # DISPLAY 2: SEPARATE BINARY & ROAD OVERLAY FOR ALL MAJOR BARS (> 3000 PX)
        # -------------------------------------------------------------------------
        if len(high_peak_indices) > 0:
            fig_peaks, axes_peaks = plt.subplots(2, len(high_peak_indices), figsize=(6 * len(high_peak_indices), 10))

            # Handle 1D array indexing when only 1 peak is present
            if len(high_peak_indices) == 1:
                axes_peaks = np.expand_dims(axes_peaks, axis=1)

            fig_peaks.suptitle(f"{edge_name}: Individual Road Feature Mapping for Bars > {pixel_threshold} Pixels",
                               fontsize=15, weight='bold')

            colors = [[255, 0, 0], [0, 255, 255], [255, 255, 0], [255, 0, 255], [0, 255, 0]]

            for col_idx, b_idx in enumerate(high_peak_indices):
                b_start = bins[b_idx]
                b_end = bins[b_idx + 1]
                b_cnt = counts[b_idx]
                bar_color = colors[col_idx % len(colors)]

                # Mask for this specific bar
                bar_mask = (angle_map >= b_start) & (angle_map < b_end) & (binary_edge_map > 0)

                # Binary Edge map for this bar
                bar_bin_img = np.zeros_like(binary_edge_map)
                bar_bin_img[bar_mask] = 255

                # Road Photo Overlay for this bar
                road_bar_img = img_rgb.copy()
                dilated_bar_mask = cv2.dilate(bar_mask.astype(np.uint8), np.ones((3, 3), np.uint8))
                road_bar_img[dilated_bar_mask > 0] = bar_color

                # Top Row: Binary Image for this bar
                axes_peaks[0, col_idx].imshow(bar_bin_img, cmap='gray')
                axes_peaks[0, col_idx].set_title(f"Bar: {b_start:.0f}°-{b_end:.0f}° | Count: {b_cnt} px\nBinary Edge Feature",
                                                  fontsize=11, weight='bold')
                axes_peaks[0, col_idx].axis('off')

                # Bottom Row: Road Photo Mapping for this bar
                axes_peaks[1, col_idx].imshow(road_bar_img)
                axes_peaks[1, col_idx].set_title(f"Mapped on Actual Roadway\n(Highlighted Feature)",
                                                  fontsize=11, weight='bold')
                axes_peaks[1, col_idx].axis('off')

            plt.tight_layout()
            plt.show()

    # Run analysis for both edge detection methods
    for name, edge_img in detectors:
        analyze_histogram_bars_on_image(name, edge_img, pixel_threshold=3000)

print("\n=========================================================")
print("ANALYSIS COMPLETE FOR ALL HISTOGRAM PEAKS & ROADWAYS.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # 7x7 Kernel

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Apply CLAHE directly on Extracted Road Grayscale
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # Helper function to compute and display Edge Map, Angle Map, Color-Coded Histogram, and Color-Matched Peak Images
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy, pixel_count_threshold=3000):
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        valid_angles = angle_map[binary_edge_map > 0]  # Consider edge pixels only

        # Connected Components
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Define distinct colors for high peaks (RGB format)
        peak_colors_rgb = [
            (255, 50, 50),     # Red
            (0, 220, 220),     # Cyan
            (255, 220, 0),     # Yellow
            (220, 50, 220),    # Magenta
            (50, 255, 50),     # Lime Green
            (255, 140, 0)      # Orange
        ]

        counts, bins = np.histogram(valid_angles.ravel(), bins=36, range=(0, 360)) if len(valid_angles) > 0 else ([], [])
        peak_bin_indices = np.where(counts > pixel_count_threshold)[0] if len(counts) > 0 else []

        # Color mapping dict for bins
        bin_color_map = {}
        for idx_enum, bin_idx in enumerate(peak_bin_indices):
            bin_color_map[bin_idx] = peak_colors_rgb[idx_enum % len(peak_colors_rgb)]

        # Plot Main Overview: Binary Edges, Angle Map, Color-Coded Histogram, and CC
        fig, ax = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Color-Coded Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Angle Map
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Orientation Angle Histogram with matched peak colors
        if len(valid_angles) > 0:
            N, bins_hist, patches = ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')

            # Color individual peak bars to match their binary highlight color
            for bin_idx, patch in enumerate(patches):
                if bin_idx in bin_color_map:
                    c_rgb = [c / 255.0 for c in bin_color_map[bin_idx]]
                    patch.set_facecolor(c_rgb)
                    patch.set_edgecolor('black')
                    patch.set_linewidth(1.5)

            # Horizontal threshold line
            ax[1, 0].axhline(y=pixel_count_threshold, color='r', linestyle='--', linewidth=1.5, label=f'Threshold ({pixel_count_threshold})')
            ax[1, 0].legend(loc='upper right')

        ax[1, 0].set_title(f"3. {edge_name} Angle Distribution (Peaks Highlighted)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")
        ax[1, 0].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

        # ==============================================================================
        # SEPARATE DISPLAY FOR EACH PEAK ANGLE RANGE WITH HISTOGRAM-MATCHED COLOR
        # ==============================================================================
        if len(peak_bin_indices) > 0:
            print(f"\n[INFO] Displaying {len(peak_bin_indices)} peak range(s) with histogram-matched colors for {edge_name}:")

            fig_peaks, axes_peaks = plt.subplots(1, len(peak_bin_indices), figsize=(6 * len(peak_bin_indices), 6))
            if len(peak_bin_indices) == 1:
                axes_peaks = [axes_peaks]

            fig_peaks.suptitle(f"{edge_name} - Separate Histogram Peak Displays (> {pixel_count_threshold} Pixels)", fontsize=15, weight='bold')

            for i, bin_idx in enumerate(peak_bin_indices):
                start_deg = bins[bin_idx]
                end_deg = bins[bin_idx + 1]
                count_val = counts[bin_idx]
                color_rgb = bin_color_map[bin_idx]

                # Create mask for this angle range
                single_peak_mask = (angle_map >= start_deg) & (angle_map < end_deg) & (binary_edge_map > 0)

                # RGB image with black background and colored pixels matching the histogram bar
                colored_peak_img = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
                colored_peak_img[single_peak_mask] = color_rgb

                # Normalize RGB for matplotlib title border box
                norm_box_color = [c / 255.0 for c in color_rgb]

                axes_peaks[i].imshow(colored_peak_img)
                title_str = f"Peak Range: {start_deg:.0f}° - {end_deg:.0f}°\nPixel Count: {count_val}"
                axes_peaks[i].set_title(title_str, fontsize=11, weight='bold',
                                         bbox=dict(boxstyle='round,pad=0.4', facecolor='black', edgecolor=norm_box_color, linewidth=2),
                                         color='white')
                axes_peaks[i].axis('off')

            plt.tight_layout()
            plt.show()
        else:
            print(f"[INFO] No peak ranges exceeded the threshold of {pixel_count_threshold} pixels for {edge_name}.")

    # Helper function for drawing standard polar Hough lines by rank slices
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"HOUGH RANKINGS ON {detector_name.upper()} EDGE MAP ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Compute base gradients for Gradient Magnitude / Canny Edge orientation
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Gradient Magnitude...")
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Gradient Magnitude", grad_binary)

    # ==============================================================================
    # STEP 5: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Canny Edge Detector...")
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y, pixel_count_threshold=3000)
    run_hough_rank_analysis("Canny Detector", canny_edges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
# Google Colab interface to upload an image from the local machine
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    # Read raw image bytes from the upload buffer into a NumPy uint8 array
    file_bytes = np.frombuffer(uploaded[filename], np.uint8)

    # Decode the image buffer into BGR color format using OpenCV
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue

    # Convert BGR (OpenCV default) to RGB for correct displaying in Matplotlib
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Convert the RGB image to HSV (Hue, Saturation, Value) color space
    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Extract only the Saturation (S) channel (Channel index 1)
    _, s_channel, _ = cv2.split(hsv_img)

    # Apply global thresholding on the Saturation channel to separate vibrant colors (vegetation) from asphalt
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)

    # Invert the binary image: asphalt/low-saturation road areas become white (255)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Display Original RGB Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    # Define a 7x7 square structuring element for morphological filtering
    kernel = np.ones((10, 10), np.uint8)

    # --- Pipeline A: Normal Saturation Morphology ---
    # Opening (Erosion followed by Dilation) removes small high-saturation noise spots
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    # Closing (Dilation followed by Erosion) fills small holes inside high-saturation regions
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    # Erosion shrinks bright regions to isolate core features
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # --- Pipeline B: Inverted Saturation Morphology (Road Region Filtering) ---
    # Opening removes isolated noise in the inverted image
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    # Closing bridges small gaps inside the extracted road area
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    # Erosion further sharpens the road boundaries
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Connected Components Analysis on inverted mask to keep large continuous road regions
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)

    # Iterate through all detected components (skipping background component index 0)
    for i in range(1, num_l_inv):
        # Keep only connected regions with pixel area larger than 1000 pixels (the main road surface)
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Visual progress of Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Visual progress of Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION + CLAHE ENHANCEMENT
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND Road Extraction and CLAHE Enhancement...")

    # Mask out non-road areas using the cleaned road binary mask
    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Extracted Road Region
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Convert isolated road image to grayscale for contrast enhancement
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Initialize CLAHE (Contrast Limited Adaptive Histogram Equalization) to enhance lane lines
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)

    # Apply Gaussian Blur to smooth high-frequency intensity noise before edge detection
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ------------------------------------------------------------------------------
    # HELPER FUNCTION 1: Plot Edge Analysis (Gradient Directions, Angles, & Blobs)
    # ------------------------------------------------------------------------------
    def process_and_plot_edge_detector(edge_name, binary_edge_map, gx, gy):
        # Calculate pixel-wise gradient angles in degrees [0°, 360°]
        angle_map = cv2.phase(gx, gy, angleInDegrees=True)
        # Select gradient angles corresponding only to active edge pixels (>0)
        valid_angles = angle_map[binary_edge_map > 0]

        # Extract connected components (blobs) formed by the binary edge map
        num_labels, labels, _, _ = cv2.connectedComponentsWithStats(binary_edge_map, connectivity=8)

        # Colorize each isolated connected component uniquely
        cc_colored = np.zeros((binary_edge_map.shape[0], binary_edge_map.shape[1], 3), dtype=np.uint8)
        np.random.seed(42)
        for i in range(1, num_labels):
            cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

        # Compute descriptive statistics of edge orientations
        if len(valid_angles) > 0:
            mean_angle = np.mean(valid_angles)
            median_angle = np.median(valid_angles)
            std_angle = np.std(valid_angles)
        else:
            mean_angle = median_angle = std_angle = 0.0

        # Plot 2x2 grid containing Binary Edges, Angle Map, Orientation Histogram, and Connected Components
        fig, ax = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"{edge_name} Analysis (Edges, Angle, Histogram & CC)", fontsize=16, weight='bold')

        # 1. Binary Edge Map
        ax[0, 0].imshow(binary_edge_map, cmap='gray')
        ax[0, 0].set_title(f"1. {edge_name} Binary Edges", fontsize=12)
        ax[0, 0].axis('off')

        # 2. Gradient Orientation Map (HSV mapped)
        ang_img = ax[0, 1].imshow(angle_map, cmap='hsv')
        ax[0, 1].set_title(f"2. {edge_name} Gradient Angle Map", fontsize=12)
        ax[0, 1].axis('off')
        fig.colorbar(ang_img, ax=ax[0, 1], fraction=0.046, pad=0.04)

        # Annotate statistical metrics on orientation map
        stats_text = (f"Mean: {mean_angle:.1f}°\n"
                      f"Median: {median_angle:.1f}°\n"
                      f"Std Dev: {std_angle:.1f}°")
        ax[0, 1].text(0.03, 0.95, stats_text, transform=ax[0, 1].transAxes,
                      fontsize=11, weight='bold', color='black',
                      verticalalignment='top',
                      bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='black'))

        # 3. Angle Distribution Histogram
        if len(valid_angles) > 0:
            counts, bins, patches = ax[1, 0].hist(valid_angles.ravel(), bins=36, range=(0, 360), color='teal', edgecolor='black')
            max_count_idx = np.argmax(counts)
            peak_angle_start = bins[max_count_idx]
            peak_angle_end = bins[max_count_idx + 1]
            peak_count = int(counts[max_count_idx])

            # Annotate peak orientation bin
            ax[1, 0].annotate(f"Peak: {peak_angle_start:.0f}°–{peak_angle_end:.0f}°\nCount: {peak_count}",
                              xy=(peak_angle_start + 5, peak_count),
                              xytext=(peak_angle_start + 25, peak_count * 0.85),
                              arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
                              fontsize=10, weight='bold', color='darkred',
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

        ax[1, 0].set_title(f"3. {edge_name} Edge Angle Distribution (0° - 360°)", fontsize=12)
        ax[1, 0].set_xlabel("Angle (Degrees)")
        ax[1, 0].set_ylabel("Pixel Count")
        ax[1, 0].grid(True, linestyle='--', alpha=0.5)

        # 4. Connected Components Visualization
        ax[1, 1].imshow(cc_colored)
        ax[1, 1].set_title(f"4. Connected Components ({num_labels - 1} Blobs)", fontsize=12)
        ax[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------------------
    # HELPER FUNCTION 2: Overlay Polar Hough Lines and Calculate Line Geometry
    # ------------------------------------------------------------------------------
    def draw_polar_line_range_with_metrics(base_img, binary_edges, lines, start_rank, end_rank, color=(255, 0, 0)):
        overlay = base_img.copy()
        if lines is None or len(lines) == 0:
            return overlay, 0, 0.0, 0.0, 0.0

        # Bound rank range indices within valid array dimensions
        actual_start = min(start_rank - 1, len(lines))
        actual_end = min(end_rank, len(lines))

        if actual_start >= actual_end:
            return overlay, 0, 0.0, 0.0, 0.0

        selected_lines = lines[actual_start:actual_end]
        count = 0
        total_length = 0.0
        total_width = 0.0
        total_thickness = 0.0

        # Loop through ranked Hough line parameters
        for line in selected_lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho

            # Project polar line parameters (rho, theta) to Cartesian endpoints across the frame
            x1 = int(x0 + 10000 * (-b))
            y1 = int(y0 + 10000 * (a))
            x2 = int(x0 - 10000 * (-b))
            y2 = int(y0 - 10000 * (a))

            # Draw polar line on the RGB overlay image
            cv2.line(overlay, (x1, y1), (x2, y2), color, 2)

            # Create line mask to evaluate active edge overlap along the line route
            mask_line = np.zeros_like(binary_edges)
            cv2.line(mask_line, (x1, y1), (x2, y2), 255, 1)
            intersection = cv2.bitwise_and(binary_edges, binary_edges, mask=mask_line)

            # Measure active edge length along the detected Hough vector
            active_pixels = np.column_stack(np.where(intersection > 0))
            if len(active_pixels) > 1:
                pts = active_pixels[:, [1, 0]]
                dists = [np.linalg.norm(pts[i] - pts[j]) for i in range(len(pts)) for j in range(i+1, len(pts))]
                line_len = max(dists) if dists else 0.0
            else:
                line_len = 0.0

            # Calculate average line width/thickness metrics
            line_area = np.sum(intersection > 0)
            line_width = line_area / line_len if line_len > 0 else 0.0
            line_thick = line_width

            total_length += line_len
            total_width += line_width
            total_thickness += line_thick
            count += 1

        avg_length = total_length / count if count > 0 else 0.0
        avg_width = total_width / count if count > 0 else 0.0
        avg_thickness = total_thickness / count if count > 0 else 0.0

        return overlay, count, avg_length, avg_width, avg_thickness

    # ------------------------------------------------------------------------------
    # HELPER FUNCTION 3: Compute Discrete Hough Parameter Accumulator Space Matrix
    # ------------------------------------------------------------------------------
    def compute_hough_accumulator(edge_map, rho_res=1, theta_res=np.pi/180):
        height, width = edge_map.shape
        # Compute maximum diagonal distance across image frame
        max_dist = int(np.ceil(np.sqrt(height**2 + width**2)))

        # Define discretization range for rho and theta parameters
        rhos = np.arange(-max_dist, max_dist, rho_res)
        thetas = np.deg2rad(np.arange(-90, 90, 1))

        cos_t = np.cos(thetas)
        sin_t = np.sin(thetas)
        num_thetas = len(thetas)

        # Allocate 2D accumulator grid (rows: Rho, cols: Theta)
        accumulator = np.zeros((2 * max_dist, num_thetas), dtype=np.uint64)

        # Extract indices of active edge pixels
        y_idxs, x_idxs = np.nonzero(edge_map)

        # Standard Hough Accumulator Voting Procedure
        # For every edge pixel (x, y), evaluate rho = x*cos(theta) + y*sin(theta) across all theta steps
        for x, y in zip(x_idxs, y_idxs):
            for t_idx in range(num_thetas):
                rho = int(round(x * cos_t[t_idx] + y * sin_t[t_idx])) + max_dist
                accumulator[rho, t_idx] += 1

        return accumulator, thetas, rhos

    # ------------------------------------------------------------------------------
    # HELPER FUNCTION 4: Run Hough Parameter Analysis & Display Heatmap + Rank Slices
    # ------------------------------------------------------------------------------
    def run_hough_rank_analysis(detector_name, edge_map):
        print(f"[EXECUTION] Running Rank-based Standard Hough Transform for {detector_name}...")
        rho_res = 1
        theta_res = np.pi / 180
        hough_threshold = 10

        # Extract standard Hough lines sorted by accumulator vote rank using OpenCV
        lines_polar = cv2.HoughLines(edge_map, rho=rho_res, theta=theta_res, threshold=hough_threshold)

        # 1. Compute and display 2D Hough Accumulator Heatmap
        accumulator, thetas, rhos = compute_hough_accumulator(edge_map)
        plt.figure(figsize=(8, 6))
        plt.imshow(accumulator, cmap='inferno', extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]], aspect='auto')
        plt.title(f"Hough Space Accumulator Heatmap - {detector_name}", fontsize=14, weight='bold')
        plt.xlabel("Theta (Degrees)", fontsize=11)
        plt.ylabel("Rho (Pixels)", fontsize=11)
        plt.colorbar(label='Vote Count / Accumulator Intensity')
        plt.tight_layout()
        plt.show()

        # 2. Slice Hough line ranks into discrete candidate groups
        rank_ranges = [
            (10, 20),
            (20, 40),
            (40, 60),
            (60, 70),
            (70, 100)
        ]

        slice_results = []
        for start, end in rank_ranges:
            img_out, lines_drawn, avg_len, avg_wid, avg_thick = draw_polar_line_range_with_metrics(
                img_rgb, edge_map, lines_polar, start_rank=start, end_rank=end
            )
            slice_results.append((start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick))

        # Display Rank Slice Line Overlays side-by-side
        fig_ranks, axes_ranks = plt.subplots(1, 5, figsize=(25, 5))
        fig_ranks.suptitle(f"HOUGH RANKINGS ON {detector_name.upper()} EDGE MAP ({filename})", fontsize=15, weight='bold')

        for idx, (start, end, img_out, lines_drawn, avg_len, avg_wid, avg_thick) in enumerate(slice_results):
            axes_ranks[idx].imshow(img_out)
            title_text = (f"Rank {start}-{end} (Lines: {lines_drawn})\n"
                          f"Len: {avg_len:.1f}px | Wid: {avg_wid:.1f}px\n"
                          f"Thick: {avg_thick:.1f}px")
            axes_ranks[idx].set_title(title_text, fontsize=9, weight='bold')
            axes_ranks[idx].axis('off')

        plt.tight_layout()
        plt.show()

    # Pre-calculate base image gradients (Horizontal and Vertical) using Sobel filters
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # ==============================================================================
    # STEP 4: SOBEL EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 4: Processing Sobel Operator...")
    # Calculate 2D gradient magnitude from Sobel X and Y derivatives
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    # Threshold normalized gradient to build binary Sobel edge map
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Sobel Operator", sobel_binary, sobel_x, sobel_y)
    run_hough_rank_analysis("Sobel Operator", sobel_binary)

    # ==============================================================================
    # STEP 5: GRADIENT MAGNITUDE
    # ==============================================================================
    print("[EXECUTION] Step 5: Processing Gradient Magnitude...")
    # Convert gradient magnitude directly to uint8 image array and apply distinct threshold
    grad_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)
    process_and_plot_edge_detector("Gradient Magnitude", grad_binary, sobel_x, sobel_y)
    run_hough_rank_analysis("Gradient Magnitude", grad_binary)

    # ==============================================================================
    # STEP 6: LAPLACIAN EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 6: Processing Laplacian Operator...")
    # Compute 2nd order spatial derivatives using Laplacian operator
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    # Threshold normalized Laplacian response to construct binary edge map
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Recompute Sobel derivatives on normalized Laplacian image to obtain gradient orientations
    lap_gx = cv2.Sobel(lap_norm, cv2.CV_64F, 1, 0, ksize=3)
    lap_gy = cv2.Sobel(lap_norm, cv2.CV_64F, 0, 1, ksize=3)
    process_and_plot_edge_detector("Laplacian Operator", lap_binary, lap_gx, lap_gy)
    run_hough_rank_analysis("Laplacian Operator", lap_binary)

    # ==============================================================================
    # STEP 7: CANNY EDGE DETECTION
    # ==============================================================================
    print("[EXECUTION] Step 7: Processing Canny Edge Detector...")
    # Run multi-stage Canny Edge Detector (includes Gaussian smoothing, gradients, NMS, and hysteresis thresholding)
    canny_edges = cv2.Canny(blurred, 50, 150)
    process_and_plot_edge_detector("Canny Detector", canny_edges, sobel_x, sobel_y)
    run_hough_rank_analysis("Canny Detector", canny_edges)

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # Set kernel > 5 (7x7)

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 4: SINGLE CONNECTED COMPONENTS ANALYSIS ON EXTRACTED ROAD
    # ==============================================================================
    print("[EXECUTION] Step 4: Computing Connected Components Analysis before Edge Detection...")

    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    _, thresholded_road = cv2.threshold(gray_masked, 1, 255, cv2.THRESH_BINARY)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresholded_road, connectivity=8)

    cc_colored = np.zeros((gray_masked.shape[0], gray_masked.shape[1], 3), dtype=np.uint8)
    np.random.seed(42)  # Fixed seed for consistent visualization colors
    for i in range(1, num_labels):
        cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()

    # Plot Step 4: Single Connected Component Visualization
    plt.figure(figsize=(8, 8))
    plt.imshow(cc_colored)
    plt.title(f"Step 4: Connected Components Analysis ({num_labels - 1} Road Regions)", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 5: EDGE DETECTION PIPELINES (SOBEL, LAPLACIAN, GRADIENT MAGNITUDE, CANNY)
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Edge Operators (Sobel, Laplacian, Gradient Magnitude, Canny)...")

    # 1. Sobel Edge Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # 2. Laplacian Edge Detection
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # 3. Gradient Magnitude Edge Detection
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    grad_magnitude = cv2.magnitude(grad_x, grad_y)
    grad_norm = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    # 4. Canny Edge Detection
    canny_edges = cv2.Canny(blurred, 50, 150)

    # Mask edge maps to ensure they remain constrained inside the extracted road region
    road_mask_bin = (s_inv_clean > 0).astype(np.uint8)
    sobel_binary = cv2.bitwise_and(sobel_binary, sobel_binary, mask=road_mask_bin)
    lap_binary = cv2.bitwise_and(lap_binary, lap_binary, mask=road_mask_bin)
    grad_binary = cv2.bitwise_and(grad_binary, grad_binary, mask=road_mask_bin)
    canny_edges = cv2.bitwise_and(canny_edges, canny_edges, mask=road_mask_bin)

    # Plot Step 5: Side-by-Side Comparison of Edge Detectors
    fig5, ax5 = plt.subplots(2, 2, figsize=(16, 14))
    fig5.suptitle("Step 5: Edge Detection Results across Connected Road Regions", fontsize=16, weight='bold')

    ax5[0, 0].imshow(sobel_binary, cmap='gray')
    ax5[0, 0].set_title("1. Sobel Edge Map", fontsize=12)
    ax5[0, 0].axis('off')

    ax5[0, 1].imshow(lap_binary, cmap='gray')
    ax5[0, 1].set_title("2. Laplacian Edge Map", fontsize=12)
    ax5[0, 1].axis('off')

    ax5[1, 0].imshow(grad_binary, cmap='gray')
    ax5[1, 0].set_title("3. Gradient Magnitude Edge Map", fontsize=12)
    ax5[1, 0].axis('off')

    ax5[1, 1].imshow(canny_edges, cmap='gray')
    ax5[1, 1].set_title("4. Canny Edge Map", fontsize=12)
    ax5[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode)...")

    kernel = np.ones((7, 7), np.uint8)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 4: CONNECTED COMPONENTS SHAPE FILTERING (FILTER OUT RECTANGLES & BORDERS)
    # ==============================================================================
    print("[EXECUTION] Step 4: Filtering out rectangular boxes, borders, and straight road markings...")

    # Adaptive threshold to isolate potential dark linear crack candidate features
    thresh_crack = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 3
    )
    thresh_crack = cv2.bitwise_and(thresh_crack, thresh_crack, mask=s_inv_clean)

    # Perform Connected Components Analysis on candidate crack features
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh_crack, connectivity=8)

    # Initialize mask specifically for irregular crack components
    crack_only_mask = np.zeros_like(thresh_crack)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        # Filter 1: Area size limits (exclude microscopic noise & large road background blocks)
        if 15 <= area <= 5000:
            bounding_box_area = w * h
            extent = float(area) / bounding_box_area if bounding_box_area > 0 else 0
            aspect_ratio = float(max(w, h)) / (min(w, h) + 1e-5)

            # Filter 2: Rectangularity / Regular Shape Filter
            # Rectangles/boxes have high extent (~1.0). Irregular cracks have low extent (< 0.5)
            # Long, thin cracks have high aspect ratio (> 1.5)
            if extent < 0.65 and (aspect_ratio > 1.3 or area < 800):
                crack_only_mask[labels == i] = 255

    # Display isolated Connected Component Cracks
    plt.figure(figsize=(7, 7))
    plt.imshow(crack_only_mask, cmap='gray')
    plt.title("Step 4: Isolated Irregular Cracks (Boxes & Borders Removed)", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 5: APPLYING EDGE DETECTORS STRICTLY TO CRACK REGIONS
    # ==============================================================================
    print("[EXECUTION] Step 5: Evaluating Sobel, Laplacian, Gradient Magnitude, and Canny on Cracks...")

    # 1. Sobel Edge Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # 2. Laplacian Edge Detection
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # 3. Gradient Magnitude Edge Detection
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    grad_magnitude = cv2.magnitude(grad_x, grad_y)
    grad_norm = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    # 4. Canny Edge Detection
    canny_edges = cv2.Canny(blurred, 50, 150)

    # Mask each edge detector output with the filtered Crack-Only Mask
    sobel_filtered = cv2.bitwise_and(sobel_binary, sobel_binary, mask=crack_only_mask)
    lap_filtered = cv2.bitwise_and(lap_binary, lap_binary, mask=crack_only_mask)
    grad_filtered = cv2.bitwise_and(grad_binary, grad_binary, mask=crack_only_mask)
    canny_filtered = cv2.bitwise_and(canny_edges, canny_edges, mask=crack_only_mask)

    # Plot Step 5: Side-by-Side Comparison of Edge Detectors on Isolated Cracks
    fig5, ax5 = plt.subplots(2, 2, figsize=(16, 14))
    fig5.suptitle("Step 5: Edge Detection Performance Comparison on Irregular Cracks", fontsize=16, weight='bold')

    ax5[0, 0].imshow(sobel_filtered, cmap='gray')
    ax5[0, 0].set_title("1. Sobel Edge Detector", fontsize=12)
    ax5[0, 0].axis('off')

    ax5[0, 1].imshow(lap_filtered, cmap='gray')
    ax5[0, 1].set_title("2. Laplacian Edge Detector", fontsize=12)
    ax5[0, 1].axis('off')

    ax5[1, 0].imshow(grad_filtered, cmap='gray')
    ax5[1, 0].set_title("3. Gradient Magnitude Detector", fontsize=12)
    ax5[1, 0].axis('off')

    ax5[1, 1].imshow(canny_filtered, cmap='gray')
    ax5[1, 1].set_title("4. Canny Edge Detector", fontsize=12)
    ax5[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # Set kernel > 5 (7x7)

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 4: IRREGULAR CRACK ISOLATION VIA CONNECTED COMPONENTS
    # ==============================================================================
    print("[EXECUTION] Step 4: Isolating Irregular Road Cracks (Filtering Straight Lines & Rectangles)...")

    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Adaptive enhancement & Black Top-Hat to highlight dark crack lines
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)

    top_hat_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    black_tophat = cv2.morphologyEx(enhanced_gray, cv2.MORPH_BLACKHAT, top_hat_kernel)

    # Threshold dark crack features
    _, crack_thresh = cv2.threshold(black_tophat, 15, 255, cv2.THRESH_BINARY)
    crack_thresh = cv2.bitwise_and(crack_thresh, crack_thresh, mask=s_inv_clean)

    # Connected Components with Geometric Shape Filtering
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(crack_thresh, connectivity=8)

    crack_only_mask = np.zeros_like(crack_thresh)
    cc_colored = np.zeros((gray_masked.shape[0], gray_masked.shape[1], 3), dtype=np.uint8)
    np.random.seed(42)

    valid_crack_count = 0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        # Calculate aspect ratio & extent (rectangularity)
        aspect_ratio = float(w) / h if h > 0 else 0
        extent = float(area) / (w * h) if (w * h) > 0 else 0

        # Filter Rules:
        # 1. Ignore small noise pixels (area < 20) and massive road borders (area > 15000)
        # 2. Ignore solid rectangular/square blocks (extent > 0.65)
        # 3. Retain elongated / irregular / low-extent crack lines
        if 20 < area < 15000:
            if extent < 0.65:  # Filters out solid rectangles and lane blocks
                crack_only_mask[labels == i] = 255
                cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()
                valid_crack_count += 1

    # Plot Step 4: Filtered Crack Connected Components
    fig4, ax4 = plt.subplots(1, 2, figsize=(16, 7))
    fig4.suptitle("Step 4: Isolated Crack Regions (Rectangular Noise & Borders Removed)", fontsize=14, weight='bold')

    ax4[0].imshow(crack_only_mask, cmap='gray')
    ax4[0].set_title("Crack Mask (Binary)")
    ax4[0].axis('off')

    ax4[1].imshow(cc_colored)
    ax4[1].set_title(f"Connected Crack Blobs ({valid_crack_count} Irregular Cracks)")
    ax4[1].axis('off')

    plt.tight_layout()
    plt.show()

    # Preprocessing Crack ROI for Edge Detectors
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 5: EDGE DETECTION BENCHMARKING ON ISOLATED CRACKS ONLY
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Edge Detectors directly on Irregular Crack Mask...")

    # 1. Sobel Edge Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # 2. Laplacian Edge Detection
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # 3. Gradient Magnitude Edge Detection
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    grad_magnitude = cv2.magnitude(grad_x, grad_y)
    grad_norm = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    # 4. Canny Edge Detection
    canny_edges = cv2.Canny(blurred, 50, 150)

    # Apply isolated crack mask to constrain all edge outputs strictly to crack zones
    sobel_crack_edges = cv2.bitwise_and(sobel_binary, sobel_binary, mask=crack_only_mask)
    lap_crack_edges = cv2.bitwise_and(lap_binary, lap_binary, mask=crack_only_mask)
    grad_crack_edges = cv2.bitwise_and(grad_binary, grad_binary, mask=crack_only_mask)
    canny_crack_edges = cv2.bitwise_and(canny_edges, canny_edges, mask=crack_only_mask)

    # Plot Step 5: Side-by-Side Comparison on Isolated Cracks
    fig5, ax5 = plt.subplots(2, 2, figsize=(16, 14))
    fig5.suptitle("Step 5: Edge Detection Comparison on Isolated Irregular Cracks", fontsize=16, weight='bold')

    ax5[0, 0].imshow(sobel_crack_edges, cmap='gray')
    ax5[0, 0].set_title("1. Sobel Crack Edges (Thick Boundary)", fontsize=12)
    ax5[0, 0].axis('off')

    ax5[0, 1].imshow(lap_crack_edges, cmap='gray')
    ax5[0, 1].set_title("2. Laplacian Crack Edges (Zero-Crossings)", fontsize=12)
    ax5[0, 1].axis('off')

    ax5[1, 0].imshow(grad_crack_edges, cmap='gray')
    ax5[1, 0].set_title("3. Gradient Magnitude Crack Edges", fontsize=12)
    ax5[1, 0].axis('off')

    ax5[1, 1].imshow(canny_crack_edges, cmap='gray')
    ax5[1, 1].set_title("4. Canny Crack Edges (Thin Single-Pixel)", fontsize=12)
    ax5[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # Set kernel > 5 (7x7)

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2a: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2b: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(21, 21))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 4: CONNECTED COMPONENTS ANALYSIS FOR IRREGULAR CRACK ISOLATION
    # ==============================================================================
    print("[EXECUTION] Step 4: Computing Connected Components Analysis on Irregular Cracks...")

    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)

    # Adaptive enhancement & Black Top-Hat to emphasize dark crack structures
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)

    top_hat_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    black_tophat = cv2.morphologyEx(enhanced_gray, cv2.MORPH_BLACKHAT, top_hat_kernel)

    # Threshold dark crack candidates strictly within the road region
    _, crack_thresh = cv2.threshold(black_tophat, 15, 255, cv2.THRESH_BINARY)
    crack_thresh = cv2.bitwise_and(crack_thresh, crack_thresh, mask=s_inv_clean)

    # Connected Components with shape filtering to remove straight lines and rectangular blocks
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(crack_thresh, connectivity=8)

    crack_only_mask = np.zeros_like(crack_thresh)
    cc_colored = np.zeros((gray_masked.shape[0], gray_masked.shape[1], 3), dtype=np.uint8)
    np.random.seed(42)  # Seed for reproducible colors

    valid_crack_count = 0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        extent = float(area) / (w * h) if (w * h) > 0 else 0

        # Filter: keep elongated/winding structures, eliminate blocky non-crack patches
        if 20 < area < 15000:
            if extent < 0.65:
                crack_only_mask[labels == i] = 255
                cc_colored[labels == i] = np.random.randint(50, 255, size=3).tolist()
                valid_crack_count += 1

    # Plot Step 4: Display Connected Components BEFORE Edge Detection
    plt.figure(figsize=(8, 8))
    plt.imshow(cc_colored)
    plt.title(f"Step 4: Connected Components Analysis ({valid_crack_count} Irregular Crack Regions)", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 5: EDGE DETECTION PIPELINES (SOBEL, LAPLACIAN, GRADIENT MAGNITUDE, CANNY)
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Edge Operators (Sobel, Laplacian, Gradient Magnitude, Canny) on Connected Crack Mask...")

    # 1. Sobel Edge Detection
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # 2. Laplacian Edge Detection
    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # 3. Gradient Magnitude Edge Detection
    grad_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    grad_magnitude = cv2.magnitude(grad_x, grad_y)
    grad_norm = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, grad_binary = cv2.threshold(grad_norm, 40, 255, cv2.THRESH_BINARY)

    # 4. Canny Edge Detection
    canny_edges = cv2.Canny(blurred, 50, 150)

    # Apply the connected component crack mask to all edge detectors
    sobel_crack_edges = cv2.bitwise_and(sobel_binary, sobel_binary, mask=crack_only_mask)
    lap_crack_edges = cv2.bitwise_and(lap_binary, lap_binary, mask=crack_only_mask)
    grad_crack_edges = cv2.bitwise_and(grad_binary, grad_binary, mask=crack_only_mask)
    canny_crack_edges = cv2.bitwise_and(canny_edges, canny_edges, mask=crack_only_mask)

    # Plot Step 5: Edge Detection Comparison on Isolated Connected Cracks
    fig5, ax5 = plt.subplots(2, 2, figsize=(16, 14))
    fig5.suptitle("Step 5: Edge Detection Results on Connected Crack Mask", fontsize=16, weight='bold')

    ax5[0, 0].imshow(sobel_crack_edges, cmap='gray')
    ax5[0, 0].set_title("1. Sobel Edge Map", fontsize=12)
    ax5[0, 0].axis('off')

    ax5[0, 1].imshow(lap_crack_edges, cmap='gray')
    ax5[0, 1].set_title("2. Laplacian Edge Map", fontsize=12)
    ax5[0, 1].axis('off')

    ax5[1, 0].imshow(grad_crack_edges, cmap='gray')
    ax5[1, 0].set_title("3. Gradient Magnitude Edge Map", fontsize=12)
    ax5[1, 0].axis('off')

    ax5[1, 1].imshow(canny_crack_edges, cmap='gray')
    ax5[1, 1].set_title("4. Canny Edge Map", fontsize=12)
    ax5[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((21, 21), np.uint8)  # Kernel size > 5

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # HELPER FUNCTION: CRACK-ONLY CONNECTED COMPONENTS FILTERING
    # ==============================================================================
    def filter_cracks_only(binary_map, min_area=15, max_area=3500, max_extent=0.55):
        """
        Filters out large rectangular shapes (cars, gantries, road borders)
        and retains thin, irregular, linear crack structures.
        """
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_map, connectivity=8)

        filtered_cracks_binary = np.zeros_like(binary_map)
        colored_cracks_cc = np.zeros((binary_map.shape[0], binary_map.shape[1], 3), dtype=np.uint8)
        crack_count = 0

        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]

            bbox_area = w * h
            extent = float(area) / bbox_area if bbox_area > 0 else 0

            # Filter logic:
            # 1. Ignore tiny isolated noise dots (area < min_area)
            # 2. Ignore huge blocky structures/cars/gantries (area > max_area)
            # 3. Ignore rectangular filled shapes (extent > max_extent)
            if min_area <= area <= max_area and extent <= max_extent:
                filtered_cracks_binary[labels == i] = 255
                colored_cracks_cc[labels == i] = np.random.randint(50, 255, size=3).tolist()
                crack_count += 1

        return filtered_cracks_binary, colored_cracks_cc, crack_count

    # ==============================================================================
    # STEP 4: SOBEL EDGE MAP + RECTANGLE-FILTERED CRACKS
    # ==============================================================================
    print("[EXECUTION] Step 4: Applying Sobel Detection & Filtering Out Rectangular Objects...")

    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Filter out rectangular boxes to focus purely on crack lines
    sobel_crack_bin, sobel_crack_cc, sobel_crack_num = filter_cracks_only(sobel_binary)

    # Plot Step 4: Sobel Filtered Cracks
    fig4, ax4 = plt.subplots(1, 2, figsize=(14, 5))
    fig4.suptitle("Step 4: Sobel Edge Map (Rectangular Boxes Removed -> Crack Focus)", fontsize=14, weight='bold')
    ax4[0].imshow(sobel_crack_bin, cmap='gray'); ax4[0].set_title("Sobel Crack Edges Only"); ax4[0].axis('off')
    ax4[1].imshow(sobel_crack_cc); ax4[1].set_title(f"Sobel Crack Connected Components ({sobel_crack_num} Lines)"); ax4[1].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 5: LAPLACIAN EDGE MAP + RECTANGLE-FILTERED CRACKS
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Laplacian Detection & Filtering Out Rectangular Objects...")

    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Filter out rectangular boxes to focus purely on crack lines
    lap_crack_bin, lap_crack_cc, lap_crack_num = filter_cracks_only(lap_binary)

    # Plot Step 5: Laplacian Filtered Cracks
    fig5, ax5 = plt.subplots(1, 2, figsize=(14, 5))
    fig5.suptitle("Step 5: Laplacian Edge Map (Rectangular Boxes Removed -> Crack Focus)", fontsize=14, weight='bold')
    ax5[0].imshow(lap_crack_bin, cmap='gray'); ax5[0].set_title("Laplacian Crack Edges Only"); ax5[0].axis('off')
    ax5[1].imshow(lap_crack_cc); ax5[1].set_title(f"Laplacian Crack Connected Components ({lap_crack_num} Lines)"); ax5[1].axis('off')
    plt.tight_layout(); plt.show()0

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("=========================================================")
print("STEP 0: UPLOADING UAV DATASET IMAGE")
print("=========================================================")
uploaded = files.upload()

for filename in uploaded.keys():
    # ==============================================================================
    # STEP 1: ORIGINAL IMAGE & HSV SATURATION EXTRACTION
    # ==============================================================================
    print(f"\n[EXECUTION] Processing file: {filename}")
    print("[EXECUTION] Step 1: Loading Original Image & Extracting Saturation Channels...")

    file_bytes = np.frombuffer(uploaded[filename], np.uint8)
    img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img_bgr is None:
        print(f"Error loading {filename}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    hsv_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, s_channel, _ = cv2.split(hsv_img)
    _, s_binary = cv2.threshold(s_channel, 50, 255, cv2.THRESH_BINARY)
    s_binary_inv = cv2.bitwise_not(s_binary)

    # Plot Step 1: Original Image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title("Step 1: Original RGB Image", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 2: SATURATION MORPHOLOGY PIPELINE (NORMAL & INVERTED)
    # ==============================================================================
    print("[EXECUTION] Step 2: Applying Morphological Operations (Open -> Close -> Erode with Kernel > 5)...")

    kernel = np.ones((7, 7), np.uint8)  # Set kernel > 5 (7x7)

    # Normal Saturation Morphology
    s_bin_open = cv2.morphologyEx(s_binary, cv2.MORPH_OPEN, kernel)
    s_bin_close = cv2.morphologyEx(s_bin_open, cv2.MORPH_CLOSE, kernel)
    s_bin_erode = cv2.erode(s_bin_close, kernel, iterations=1)

    # Inverted Saturation Morphology
    s_inv_open = cv2.morphologyEx(s_binary_inv, cv2.MORPH_OPEN, kernel)
    s_inv_close = cv2.morphologyEx(s_inv_open, cv2.MORPH_CLOSE, kernel)
    s_inv_erode = cv2.erode(s_inv_close, kernel, iterations=1)

    # Area filter on inverted mask to retain main road area
    num_l_inv, lab_inv, st_inv, _ = cv2.connectedComponentsWithStats(s_inv_erode, connectivity=8)
    s_inv_clean = np.zeros_like(s_inv_erode)
    for i in range(1, num_l_inv):
        if st_inv[i, cv2.CC_STAT_AREA] > 1000:
            s_inv_clean[lab_inv == i] = 255

    # Plot Step 2: Normal Saturation Morphology
    fig2a, ax2a = plt.subplots(1, 4, figsize=(20, 5))
    fig2a.suptitle("Step 2a: Normal Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2a[0].imshow(s_binary, cmap='gray'); ax2a[0].set_title("1. Raw Binary"); ax2a[0].axis('off')
    ax2a[1].imshow(s_bin_open, cmap='gray'); ax2a[1].set_title("2. Morph Open"); ax2a[1].axis('off')
    ax2a[2].imshow(s_bin_close, cmap='gray'); ax2a[2].set_title("3. Morph Close"); ax2a[2].axis('off')
    ax2a[3].imshow(s_bin_erode, cmap='gray'); ax2a[3].set_title("4. Erode"); ax2a[3].axis('off')
    plt.tight_layout(); plt.show()

    # Plot Step 2: Inverted Saturation Morphology
    fig2b, ax2b = plt.subplots(1, 4, figsize=(20, 5))
    fig2b.suptitle("Step 2b: Inverted Saturation Binary Morphological Pipeline", fontsize=14, weight='bold')
    ax2b[0].imshow(s_binary_inv, cmap='gray'); ax2b[0].set_title("1. Inverted Binary"); ax2b[0].axis('off')
    ax2b[1].imshow(s_inv_open, cmap='gray'); ax2b[1].set_title("2. Morph Open"); ax2b[1].axis('off')
    ax2b[2].imshow(s_inv_close, cmap='gray'); ax2b[2].set_title("3. Morph Close"); ax2b[2].axis('off')
    ax2b[3].imshow(s_inv_clean, cmap='gray'); ax2b[3].set_title("4. Cleaned Road Mask"); ax2b[3].axis('off')
    plt.tight_layout(); plt.show()

    # ==============================================================================
    # STEP 3: BITWISE_AND ROAD EXTRACTION
    # ==============================================================================
    print("[EXECUTION] Step 3: Applying Bitwise_AND using Inverted Cleaned Saturation Mask...")

    rgb_masked_output = cv2.bitwise_and(img_rgb, img_rgb, mask=s_inv_clean)

    # Plot Step 3: Bitwise Output
    plt.figure(figsize=(7, 7))
    plt.imshow(rgb_masked_output)
    plt.title("Step 3: Bitwise_AND Extracted Road Output", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # Preprocessing for Edge Detection
    gray_masked = cv2.cvtColor(rgb_masked_output, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_gray = clahe.apply(gray_masked)
    blurred = cv2.GaussianBlur(enhanced_gray, (3, 3), 0)

    # ==============================================================================
    # STEP 4: SOBEL EDGE MAP
    # ==============================================================================
    print("[EXECUTION] Step 4: Applying Sobel Edge Detection on Extracted Road...")

    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobel_x, sobel_y)
    sobel_abs = cv2.convertScaleAbs(sobel_mag)
    sobel_norm = cv2.normalize(sobel_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, sobel_binary = cv2.threshold(sobel_norm, 30, 255, cv2.THRESH_BINARY)

    # Plot Step 4: Sobel Edges
    plt.figure(figsize=(7, 7))
    plt.imshow(sobel_binary, cmap='gray')
    plt.title("Step 4: Sobel Edge Map", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

    # ==============================================================================
    # STEP 5: LAPLACIAN EDGE MAP
    # ==============================================================================
    print("[EXECUTION] Step 5: Applying Laplacian Edge Detection on Extracted Road...")

    lap_64f = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    lap_abs = cv2.convertScaleAbs(lap_64f)
    lap_norm = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)
    _, lap_binary = cv2.threshold(lap_norm, 20, 255, cv2.THRESH_BINARY)

    # Plot Step 5: Laplacian Edges
    plt.figure(figsize=(7, 7))
    plt.imshow(lap_binary, cmap='gray')
    plt.title("Step 5: Laplacian Edge Map", fontsize=14, weight='bold')
    plt.axis('off')
    plt.show()

print("\n=========================================================")
print("PROCESSING COMPLETE FOR ALL UPLOADED IMAGES.")
print("=========================================================")